# Privacy-Budget-Aware, Risk-Gated Adaptive Continuous Authentication
### Experimental study for the PhD dissertation *"Multi-Modal Biometric Fusion for Adaptive Continuous Authentication in National Digital Identity Systems"* (A. K. Aseres, AASTU)

**Primary dissertation question:** RQ4 — how privacy-enhancing techniques affect the functionality, security and performance of biometric authentication. **Secondary:** RQ3 — adaptive models that accommodate behavioural change while maintaining security.

**Refined research question.** In a continuous-authentication (CA) system whose user model is updated online, does (i) participant-level differentially private training with an explicit privacy budget and (ii) a risk-gated update policy that separates the *access* decision from the *update* decision, allow the system to track legitimate behavioural drift while limiting the damage an impostor can cause by poisoning the update stream — and what does each mechanism cost in utility, user friction and privacy budget?

**Revision (November 2026).** Sections 1–19 are the original study, unchanged except for the three documented fixes listed below; Section 20 adds every experiment requested by the IEEE Access reviewers (Access-2026-55189). Fixes: (i) `paired_test` now rounds subject-level differences to 1e-10 before identifying ties, as the manuscript's methods state (the uploaded notebook did not, which is why its primary p-value was 0.299 while the manuscript reported 0.328; with the rounding the notebook reproduces 0.328 and the 17/26/4 split); (ii) hypothesis verdicts now carry a margin-aware interpretation (Reviewer 1, point 2); (iii) a stray literal "\n" in the limitations cell.

This notebook is restart-and-run reproducible. Every number in the results sections is produced by the code below; nothing is typed in by hand. Sections follow the required 19-part structure.

## 1. Objective, hypotheses and contributions

**Proposed method (P).** A per-user logistic verifier trained by *unit-level DP-SGD*: each background enrollee (all of their samples) is one privacy unit and each of the user's own samples is one unit; per-unit gradients are clipped, units are Poisson-sampled and Gaussian noise is added. A privacy accountant (Google `dp_accounting`, RDP; PLD reported as a cross-check) tracks the cumulative ε over the initial training *and every later update*. Access and update are separate decisions:

* **Access:** accept silently if the calibrated score p ≥ τ_acc; otherwise challenge with a strong factor (step-up). A genuine user who passes is *verified*; a failed challenge locks the session.
* **Update (risk gate):** strong-factor-verified samples are always eligible. A *silently accepted (unlabelled)* sample is eligible only if (a) p ≥ τ_upd > τ_acc, (b) its diagonal-Mahalanobis novelty relative to the **enrolment anchor** is inside a fixed trust radius, and (c) its episode shows consistent evidence (≥ρ accepted, median p ≥ τ_upd). Admitted samples queue until ≥ m_min are available, the privacy budget can pay for an update, and a **budget-pacing** rule spreads the affordable updates across the deployment horizon.

**Pre-registered hypotheses** (frozen in `outputs/preregistration.json` before the main experiment; the pilot partition, seed 999, is excluded from all analyses):

| ID | Hypothesis | Endpoint | Test |
|---|---|---|---|
| **H2 (primary)** | The proposed method P reduces poisoning-induced gain in impostor acceptance relative to unrestricted self-update B1 | Attack gain = PSR(C3) − PSR(C0): acceptance of the attacker's held-out natural samples by the final model, same victim/seed, white-box confidence-aware attack vs. drift-only | Two-sided Wilcoxon signed-rank over subjects (seed-averaged), α = 0.05; 95% subject-cluster bootstrap CI |
| H1 | DP costs utility | Late-session prequential EER (C0): B0_DP vs B0, P vs B3 | Wilcoxon, Holm |
| H3 | P tracks drift better than a static model | Online genuine challenge rate (C0): P vs B0 | Wilcoxon, Holm |
| H4 | Gate and DP interact | Attack gain: B3 vs B1 (gate w/o DP), P vs B2 (gate under DP), P vs B3 (DP under gate), interaction (B2−P)−(B1−B3) | Wilcoxon, Holm |
| H5 | The gate's silent admissions add value over strong-factor-only (labels-only) updating | Online challenge rate (C0) and attack gain: P vs B2c, B3 vs B1c | Wilcoxon, Holm |

Practical-significance margin: 5 percentage points for rate metrics. Negative and null results are reported as such.

**Contributions (claimed only to the extent the results below support them):** (1) a mechanism that couples a formal participant-level privacy budget to the *update schedule* of an adaptive CA model (budget pacing); (2) an anchored trust region that bounds how far unlabelled admissions can move the model; (3) a factorial evaluation separating the effects of DP, gating and their interaction under drift and four attack conditions, with subject-clustered statistics.

## 2. Verified literature and closest competing work

Full literature matrix is in the accompanying research-design document. Closest work and how this study differs:

| Work | What it does | Gap relative to this study |
|---|---|---|
| Biggio et al., *Poisoning Adaptive Biometric Systems*, SSPR 2012, doi:10.1007/978-3-642-34166-3_46 | Gradual template-poisoning against self-update | Face, no privacy mechanism, no step-up |
| Lovisotto et al., *Biometric Backdoors*, IEEE EuroS&P 2020 (arXiv:1905.09162) | Poisoning of unsupervised template update + poisoning detector | Face/deep embeddings; no DP; no behavioural drift stream |
| Mhenni et al., *Double serial adaptation…*, Computers & Security 83 (2019), doi:10.1016/j.cose.2019.02.002; Mhenni et al., CW 2019 (attacks on adaptive keystroke strategies) | Template + threshold adaptation for keystroke dynamics; vulnerability of adaptive strategies | No privacy mechanism; no evidence/novelty gate; no step-up labels |
| Pisani et al., *Enhanced template update*, Computers & Security 60 (2016), doi:10.1016/j.cose.2016.04.004 | Uses genuine + impostor-classified samples for update | No adversarial poisoning or privacy |
| SBSeg 2024 short paper on user-specific thresholds in adaptive keystroke systems (CMU + KeyRecs streams) | Stream protocol with injected impostors | Random (non-adaptive) impostors; no privacy |
| Ma, Zhu & Hsu, IJCAI 2019, doi:10.24963/ijcai.2019/657 | DP learners resist poisoning of few items; protection degrades with more poisoned items | Generic learners, offline; not adaptive CA |
| Frontiers in AI 2026, doi:10.3389/frai.2026.1914353 | Mouse dynamics with FL + DP + adversarial robustness | Static training; no update-stream poisoning; no budget-aware updating |
| Baig et al., Int. J. Inf. Secur. 22 (2023), doi:10.1007/s10207-023-00721-y | Cryptographic privacy-preserving CA protocols | Protects matching, not what the trained model reveals; no adaptation |
| Parkinson et al., arXiv:2609.29851 (2026, preprint) | Measures keystroke template ageing over 8 weeks | Measurement only; no adaptation/privacy |

## 3. Dataset acquisition, licence and access notes

| Role | Dataset | Source used here | Access status | Licence / terms |
|---|---|---|---|---|
| **Primary** | CMU Keystroke Dynamics Benchmark (Killourhy & Maxion, DSN 2009): 51 subjects × 8 sessions × 50 repetitions of `.tie5Roanl`, sessions ≥1 day apart | Official host `cs.cmu.edu/~keystroke` was **unreachable from the execution sandbox**; two independent GitHub mirrors used and cross-checked (numerically identical) | Public | Released by the authors for research with a citation request. **Confirm the current terms on the official page before publication.** |
| **External validation** | Touchalytics (Frank et al., IEEE TIFS 2013): 41 users, 4 phones; a subset returned ~1 week later | GitHub mirror of the authors' `featMat.csv` + raw `data.csv` (raw timestamps used to verify the inter-week gap) | Public mirror | Original authors' research release; mirror repository's MIT licence covers its code, not the data. **Confirm terms with the original source.** |
| Considered, not run | Parkinson et al. 8-week keystroke data (IET Biometrics 2023; GitHub `sparkins01/keystroke`) | Reachable | README restricts use to verification/replication of the accompanying paper | Not used without the authors' permission (see manifest) |
| Considered, not reachable here | HMOG (W&M), BB-MAS (IEEE DataPort, login), KeyRecs (Zenodo), Aalto/KVC | Hosts blocked from the sandbox or registration required | — | The pipeline accepts any `{subject: (X, session)}` dict; see Section 19 |

Raw URLs are pinned to commit hashes and verified by SHA-256 below.

## 4. Environment setup

In [ ]:
import sys, subprocess, importlib, os, json, time, hashlib, platform, datetime
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, m in [('dp-accounting', 'dp_accounting'), ('pandas', None), ('scipy', None), ('matplotlib', None), ('scikit-learn', 'sklearn')]:
    ensure(p, m)
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, scipy, sklearn, dp_accounting
matplotlib.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
ROOT = os.path.abspath(os.environ.get('PACA_ROOT', '.'))
DATA, OUT, FIG = [os.path.join(ROOT, d) for d in ('data', 'outputs', 'outputs/figures')]
for d in (DATA, OUT, FIG): os.makedirs(d, exist_ok=True)
FORCE_RERUN = os.environ.get('PACA_FORCE', '0') == '1'      # set to 1 to ignore cached result CSVs
SEEDS_MAIN = [0, 1, 2, 3, 4]; SEEDS_SENS = [0, 1]; PILOT_SEED = 999   # pilot seed is never analysed
SMOKE = os.environ.get('PACA_SMOKE', '0') == '1'   # quick wiring test only; never used for reported results
MAX_VICTIMS = 3 if SMOKE else None
if SMOKE: SEEDS_MAIN, SEEDS_SENS = [0, 1], [0]
versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'matplotlib': matplotlib.__version__, 'scikit-learn': sklearn.__version__,
            'dp-accounting': importlib.metadata.version('dp-accounting'), 'platform': platform.platform()}
print(json.dumps(versions, indent=1)); print('ROOT =', ROOT)

### Core library (tested module, written to disk so it can be imported and reused)

In [ ]:
%%writefile pacau_core.py
# ==== pacau_core: Privacy-budget-Aware, risk-gated Continuous AUthentication ====
import numpy as np, pandas as pd, time, logging, warnings
logging.disable(logging.WARNING)
from dp_accounting import dp_event, rdp, pld

# ---------------------------------------------------------------- data
CMU_FEATURES = None

def load_cmu(path):
    df = pd.read_csv(path)
    assert df.shape == (20400, 34), df.shape
    assert df.isna().sum().sum() == 0
    df = df.sort_values(['subject', 'sessionIndex', 'rep']).reset_index(drop=True)
    feats = [c for c in df.columns if c not in ('subject', 'sessionIndex', 'rep')]
    data = {}
    for s, d in df.groupby('subject', sort=True):
        data[s] = dict(X_raw=d[feats].to_numpy(float), sess=d['sessionIndex'].to_numpy(int))
    return data, feats, df

def cmu_transform(X):
    # a-priori, data-independent transform: signed log of timing in 10 ms units
    return np.sign(X) * np.log1p(np.abs(X) / 0.01)

TOUCH_COLS = ['user', 'doc', 'inter_stroke_time', 'stroke_duration', 'start_x', 'start_y', 'stop_x', 'stop_y',
              'direct_e2e_dist', 'mean_resultant_length', 'direction_flag', 'e2e_direction', 'phone_id',
              'vel_p20', 'vel_p50', 'vel_p80', 'acc_p20', 'acc_p50', 'acc_p80', 'med_vel_last3',
              'max_dev_e2e', 'dev_p20', 'dev_p50', 'dev_p80', 'avg_direction', 'traj_length', 'ratio_e2e_traj',
              'avg_velocity', 'med_acc_first5', 'mid_pressure', 'mid_area', 'mid_finger_orient',
              'finger_orient_change', 'phone_orientation']

def touch_transform(X):
    return np.sign(X) * np.log1p(np.abs(X))

def load_touch(feat_path, raw_path, window=5):
    f = pd.read_csv(feat_path, header=None, low_memory=False)
    f = f.apply(pd.to_numeric, errors='coerce')          # non-numeric cells (header, corrupted rows) -> NaN
    f.columns = TOUCH_COLS
    f = f.replace([np.inf, -np.inf], np.nan)
    n_nan = int(f.isna().any(axis=1).sum()); f = f.dropna()
    r = pd.read_csv(raw_path, header=None)
    r.columns = ['phone', 'user', 'doc', 't', 'action', 'orient', 'x', 'y', 'pressure', 'area', 'forient']
    start = r.groupby(['user', 'doc']).t.min().reset_index()
    start['user'] = start['user'] + 1                    # verified: featMat ids = raw ids + 1 (doc-presence patterns match)
    feats = [c for c in TOUCH_COLS if c not in ('user', 'doc', 'phone_id')]
    data, meta = {}, []
    for u, d in f.groupby('user', sort=True):
        u = int(u)
        st = start[start.user == u].copy(); st['doc_feat'] = st['doc'] + 1   # raw doc 0..6 == featMat doc 1..7
        order = st.sort_values('t')
        docs_in_order = [int(x) for x in order.doc_feat if x in set(d.doc.astype(int))]
        t0 = order.t.min()
        Xs, sess = [], []
        for k, doc in enumerate(docs_in_order):
            dd = d[d.doc == doc][feats].to_numpy(float)          # stroke order within doc preserved
            nw = len(dd) // window
            if nw == 0: continue
            W = dd[:nw * window].reshape(nw, window, -1).mean(1)
            Xs.append(W); sess += [k + 1] * nw
            tdoc = float(order[order.doc_feat == doc].t.iloc[0])
            meta.append(dict(user=u, doc=doc, order=k + 1, n_windows=nw, days_from_first=(tdoc - t0) / 864e5))
        data[u] = dict(X_raw=np.vstack(Xs), sess=np.array(sess))
    return data, feats, pd.DataFrame(meta), n_nan

# ---------------------------------------------------------------- privacy accounting
class Accountant:
    """Wraps Google dp_accounting. Cumulative composition of Poisson-subsampled Gaussian steps."""
    def __init__(self, q, sigma, delta):
        self.q, self.sigma, self.delta = q, sigma, delta
        self._cache = {}
    def eps(self, steps, method='rdp'):
        if steps == 0: return 0.0
        key = (steps, method)
        if key not in self._cache:
            ev = dp_event.PoissonSampledDpEvent(self.q, dp_event.GaussianDpEvent(self.sigma))
            a = rdp.RdpAccountant() if method == 'rdp' else pld.PLDAccountant()
            a.compose(ev, int(steps)); self._cache[key] = float(a.get_epsilon(self.delta))
        return self._cache[key]

# ---------------------------------------------------------------- model + (DP-)SGD
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def add_bias(X): return np.hstack([X, np.ones((len(X), 1))])

def sgd_train(w, Xb, y, unit, n_units, steps, lr, rng, q, clip=None, sigma=0.0, l2=1e-3, stats=None):
    """Unit-level (DP-)SGD for logistic regression.
    Each privacy unit u contributes g_u = mean_{i in u} grad_i ; g_u is clipped to norm <= clip ;
    units are Poisson-sampled with rate q ; Gaussian noise N(0, (sigma*clip)^2 I) is added to the sum ;
    the noisy sum is divided by the expected batch size q*n_units.  clip=None, sigma=0 -> non-private
    reference with the identical optimiser."""
    d = Xb.shape[1]
    cnt = np.bincount(unit, minlength=n_units).astype(float)
    exp_b = max(q * n_units, 1.0)
    for _ in range(steps):
        sel_u = rng.random(n_units) < q
        m = sel_u[unit]
        if m.any():
            Xs, ys, us = Xb[m], y[m], unit[m]
            g_i = (sigmoid(Xs @ w) - ys)[:, None] * Xs
            G = np.zeros((n_units, d))
            np.add.at(G, us, g_i)
            G = G[sel_u] / cnt[sel_u][:, None]
            if clip is not None:
                nrm = np.linalg.norm(G, axis=1)
                if stats is not None: stats.append(float(nrm.max()))
                G = G * np.minimum(1.0, clip / np.maximum(nrm, 1e-12))[:, None]
            g = G.sum(0)
        else:
            g = np.zeros(d)
        if sigma > 0: g = g + rng.normal(0.0, sigma * clip, size=d)
        w = w - lr * (g / exp_b + l2 * w)
    return w

def platt_fit(z, y, iters=100, ridge=1e-2):
    """Class-balanced Platt scaling with Platt's (1999) smoothed targets and a ridge penalty on the
    slope, solved by damped Newton steps; avoids divergence on (near-)separable calibration sets."""
    npos, nneg = max(y.sum(), 1), max((1 - y).sum(), 1)
    t = np.where(y == 1, (npos + 1) / (npos + 2), 1 / (nneg + 2))
    wts = np.where(y == 1, 0.5 / npos, 0.5 / nneg)
    a, b = 1.0, 0.0
    for _ in range(iters):
        p = sigmoid(a * z + b); r = wts * (p - t); s_ = wts * p * (1 - p) + 1e-12
        g = np.array([np.sum(r * z) + ridge * (a - 1.0), np.sum(r)])
        H = np.array([[np.sum(s_ * z * z) + ridge, np.sum(s_ * z)], [np.sum(s_ * z), np.sum(s_) + 1e-9]])
        step = np.linalg.solve(H, g); step = step * min(1.0, 2.0 / (np.abs(step).max() + 1e-12))
        a, b = a - step[0], b - step[1]
        if np.abs(step).max() < 1e-7: break
    return float(a), float(b)

def eer(gen, imp):
    """Threshold-free EER (tie-aware: thresholds placed only at distinct score values)."""
    thr = np.unique(np.concatenate([gen, imp]))
    frr = np.searchsorted(np.sort(gen), thr, side='left') / len(gen)          # gen < thr rejected
    far = 1 - np.searchsorted(np.sort(imp), thr, side='left') / len(imp)      # imp >= thr accepted
    i = np.argmin(np.abs(frr - far)); return float((frr[i] + far[i]) / 2)

def auc(pos, neg):
    from scipy.stats import mannwhitneyu
    if len(pos) == 0 or len(neg) == 0: return np.nan
    return float(mannwhitneyu(pos, neg).statistic / (len(pos) * len(neg)))

# ---------------------------------------------------------------- authenticator
DEFAULT_CFG = dict(
    n_enroll=40, n_cal=10, n_bg=50, q=0.2, sigma=3.0, clip=1.0, delta=1e-5, lr=0.05, l2=1e-3,
    init_steps=150, upd_steps=25, buffer_cap=100, W=10, tau_acc=0.5, tau_upd=0.8,
    rho=0.8, m_min=5, trust_q=0.95, eps_budget=8.0, burst=10, attack_holdout=0.3,
    p_gen_pass=1.0, p_att_pass=0.0, recal_verified=20,                       # strong-factor (step-up) outcome model (assumption A1)
    adapt_step_up=0.15, adapt_step_down=0.10, n_leak=5, probe_per_subject=20,
)

# silent: how silently-accepted (unlabelled) samples may enter the model
#   'all' = every accepted sample; 'double' = p>=tau_upd; 'risk' = proposed gate; 'none' = never
# labels: whether strong-factor-verified samples (passed step-up) are admissible
METHODS = {
  'B0_static':         dict(adaptive=False, dp=False, silent='none',   labels=False, budget=False, pacing=False, novelty=False, evidence=False, calib=True),
  'B0_static_DP':      dict(adaptive=False, dp=True,  silent='none',   labels=False, budget=False, pacing=False, novelty=False, evidence=False, calib=True),
  'B1_selfupdate':     dict(adaptive=True,  dp=False, silent='all',    labels=True,  budget=False, pacing=False, novelty=False, evidence=False, calib=True),
  'B1b_double_thr':    dict(adaptive=True,  dp=False, silent='double', labels=True,  budget=False, pacing=False, novelty=False, evidence=False, calib=True),
  'B1c_labels_only':   dict(adaptive=True,  dp=False, silent='none',   labels=True,  budget=False, pacing=False, novelty=False, evidence=False, calib=True),
  'B2_DP_selfupdate':  dict(adaptive=True,  dp=True,  silent='all',    labels=True,  budget=True,  pacing=False, novelty=False, evidence=False, calib=True),
  'B2c_DP_labels_only':dict(adaptive=True,  dp=True,  silent='none',   labels=True,  budget=True,  pacing=True,  novelty=False, evidence=False, calib=True),
  'B3_gate_nonDP':     dict(adaptive=True,  dp=False, silent='risk',   labels=True,  budget=False, pacing=False, novelty=True,  evidence=True,  calib=True),
  'P_gate_DP_budget':  dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=True,  pacing=True,  novelty=True,  evidence=True,  calib=True),
  'A_P_no_budget':     dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=False, pacing=False, novelty=True,  evidence=True,  calib=True),
  'A_P_no_pacing':     dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=True,  pacing=False, novelty=True,  evidence=True,  calib=True),
  'A_P_no_novelty':    dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=True,  pacing=True,  novelty=False, evidence=True,  calib=True),
  'A_P_no_evidence':   dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=True,  pacing=True,  novelty=True,  evidence=False, calib=True),
  'A_P_no_calib':      dict(adaptive=True,  dp=True,  silent='risk',   labels=True,  budget=True,  pacing=True,  novelty=True,  evidence=True,  calib=False),
}

class Authenticator:
    def __init__(self, Xg_anchor, Xbg_units, Xcal_g, Xcal_bg, cfg, method, acct, rng):
        self.c, self.m, self.acct, self.rng = cfg, method, acct, rng
        self.anchor = Xg_anchor; self.buf = np.zeros((0, Xg_anchor.shape[1])); self.bufflag = np.zeros(0, bool)
        self.bg = Xbg_units                      # list of arrays, one per background participant
        self.Xcal = np.vstack([Xcal_g] + [Xcal_bg]); self.ycal = np.r_[np.ones(len(Xcal_g)), np.zeros(len(Xcal_bg))]
        self.steps = 0; self.n_updates = 0; self.w = np.zeros(Xg_anchor.shape[1] + 1)
        # anchored trust region (diagonal Mahalanobis on enrolment data, computed on-device)
        G = np.vstack([Xg_anchor, Xcal_g]); self.nmu = Xg_anchor.mean(0); self.nsd = Xg_anchor.std(0) + 1e-6
        self.R = np.quantile(self.novelty(G), cfg['trust_q']) if method['novelty'] else np.inf
        self.clip_stats = [] if method['dp'] else None
        self.pend = np.zeros((0, Xg_anchor.shape[1])); self.pendflag = np.zeros(0, bool)
        self.last_update_t = 0; self.verified = np.zeros((0, Xg_anchor.shape[1]))
    def novelty(self, X): return np.sqrt(np.mean(((X - self.nmu) / self.nsd) ** 2, axis=1))
    def _dataset(self):
        Xs = list(self.bg) + [np.vstack([self.anchor, self.buf])]
        nb = len(self.bg); G = Xs[-1]
        X = np.vstack(Xs); y = np.r_[np.zeros(sum(len(b) for b in self.bg)), np.ones(len(G))]
        unit = np.r_[np.concatenate([np.full(len(b), i) for i, b in enumerate(self.bg)]), nb + np.arange(len(G))].astype(int)
        return add_bias(X), y, unit, nb + len(G)
    def fit(self, steps):
        Xb, y, unit, nu = self._dataset(); c = self.c
        if self.m['dp']:
            self.w = sgd_train(self.w, Xb, y, unit, nu, steps, c['lr'], self.rng, c['q'], clip=c['clip'], sigma=c['sigma'], l2=c['l2'], stats=self.clip_stats)
        else:
            self.w = sgd_train(self.w, Xb, y, unit, nu, steps, c['lr'], self.rng, c['q'], clip=None, sigma=0.0, l2=c['l2'])
        self.steps += steps
        Xc, yc = self.Xcal, self.ycal
        if self.m['labels'] and len(self.verified) and self.c['recal_verified'] > 0:
            V = self.verified[-self.c['recal_verified']:]
            Xc, yc = np.vstack([Xc, V]), np.r_[yc, np.ones(len(V))]
        self.a, self.b = platt_fit(add_bias(Xc) @ self.w, yc) if self.m['calib'] else (1.0, 0.0)
    def eps(self, steps=None):
        return self.acct.eps(self.steps if steps is None else steps) if self.m['dp'] else np.inf
    def n_affordable(self):
        k = 0
        while self.acct.eps(self.steps + (k + 1) * self.c['upd_steps']) <= self.c['eps_budget'] and k < 10000: k += 1
        return k
    def can_spend(self, t, horizon):
        if not (self.m['dp'] and self.m['budget']): return True
        if self.acct.eps(self.steps + self.c['upd_steps']) > self.c['eps_budget']: return False
        if self.m['pacing']:                                   # spread the remaining budget over the horizon
            if not hasattr(self, 'min_gap'): self.min_gap = horizon / max(self.n_affordable(), 1)
            return (t - self.last_update_t) >= self.min_gap
        return True
    def score(self, X): return sigmoid(self.a * (add_bias(X) @ self.w) + self.b)
    def admit(self, X, p, verified):
        """Update eligibility for one episode -- a decision separate from access.
        verified[i] = True if sample i was challenged and passed the strong factor (labelled genuine)."""
        c, m = self.c, self.m
        lab = verified & m['labels']
        sil = (p >= c['tau_acc']) & ~verified                      # silently accepted, unlabelled
        if m['silent'] == 'none': ok_s = np.zeros(len(p), bool)
        elif m['silent'] == 'all': ok_s = sil
        elif m['silent'] == 'double': ok_s = sil & (p >= c['tau_upd'])
        else:                                                      # proposed risk gate
            ok_s = sil & (p >= c['tau_upd']) & (self.novelty(X) <= self.R)
            if m['evidence'] and not ((np.mean(p >= c['tau_acc']) >= c['rho']) and (np.median(p) >= c['tau_upd'])):
                ok_s[:] = False
        return lab | ok_s
    def add_verified(self, x): self.verified = np.vstack([self.verified, x])[-200:]
    def queue(self, Xnew, flags):
        self.pend = np.vstack([self.pend, Xnew]); self.pendflag = np.r_[self.pendflag, flags]
    def maybe_update(self, t, horizon, mmin):
        if len(self.pend) < mmin or not self.can_spend(t, horizon): return False
        self.buf = np.vstack([self.buf, self.pend])[-self.c['buffer_cap']:]
        self.bufflag = np.r_[self.bufflag, self.pendflag][-self.c['buffer_cap']:]
        self.pend = self.pend[:0]; self.pendflag = self.pendflag[:0]
        self.fit(self.c['upd_steps']); self.n_updates += 1; self.last_update_t = t
        return True

# ---------------------------------------------------------------- scenario construction (leakage-safe)
def sid(s):
    """Stable integer id for subject labels such as 's046' or 41."""
    d = ''.join(ch for ch in str(s) if ch.isdigit()); return int(d) if d else abs(hash(str(s))) % 10**6

def partition(subjects, n_bgtrain, n_bgcal, n_att, seed):
    r = np.random.default_rng(seed); s = np.array(sorted(subjects)); r.shuffle(s)
    a, b, c = n_bgtrain, n_bgtrain + n_bgcal, n_bgtrain + n_bgcal + n_att
    return dict(bg_train=list(s[:a]), bg_cal=list(s[a:b]), attackers=list(s[b:c]), victims=list(s[c:]))

def build_run(data, transform, part, victim, attacker, cfg, seed):
    """All samples of one (seed, victim) run. Standardiser + Platt reference = bg_cal (public reference set)."""
    r = np.random.default_rng(seed * 100003 + sid(victim))
    T = lambda s: transform(data[s]['X_raw'])
    ref = np.vstack([T(s)[r.choice(len(data[s]['sess']), min(cfg['n_bg'], len(data[s]['sess'])), replace=False)] for s in part['bg_cal']])
    mu, sd = ref.mean(0), ref.std(0) + 1e-6
    Z = lambda X: (X - mu) / sd
    Xv, sv = Z(T(victim)), data[victim]['sess']
    ne, nc = cfg['n_enroll'], cfg['n_cal']
    run = dict(victim=victim, attacker=attacker, mu=mu, sd=sd, Z=Z,
               anchor=Xv[:ne], cal_g=Xv[ne:ne + nc], stream_X=Xv[ne + nc:], stream_s=sv[ne + nc:],
               cal_bg=Z(ref))
    run['bg_units'] = [Z(T(s)[r.choice(len(data[s]['sess']), min(cfg['n_bg'], len(data[s]['sess'])), replace=False)]) for s in part['bg_train']]
    Xa_raw = data[attacker]['X_raw']; na = len(Xa_raw); cut = int(round(na * (1 - cfg['attack_holdout'])))
    run['att_pool_raw'] = Xa_raw[:cut]; run['att_hold'] = Z(transform(Xa_raw[cut:]))
    run['leak_raw'] = data[victim]['X_raw'][:ne][r.choice(ne, cfg['n_leak'], replace=False)]
    probes = [p for p in part['attackers'] if p != attacker]
    run['probe'] = np.vstack([Z(T(p))[r.choice(len(data[p]['sess']), cfg['probe_per_subject'], replace=False)] for p in probes])
    run['transform'] = transform
    return run

def attack_plan(n_stream, rate, burst, seed):
    """Burst start positions (indices into the genuine stream before which a burst is inserted)."""
    if rate <= 0: return []
    r = np.random.default_rng(seed); nb = int(round(rate * n_stream / burst))
    lo = min(10, n_stream // 10)
    return sorted(r.choice(np.arange(lo, n_stream), nb, replace=False).tolist())

CONDITIONS = {'C0_drift_only': ('none', 0.0), 'C1_zero_effort_10': ('zero', 0.10),
              'C2_zero_effort_30': ('zero', 0.30), 'C3_confaware_oracle_20': ('oracle', 0.20),
              'C4_greybox_feedback_20': ('adaptive', 0.20)}
ALPHA_GRID = np.linspace(0, 1, 21)

def run_stream(run, cfg, method, acct, cond, seed, collect_trace=False):
    """Prequential (test-then-train) simulation of one victim's chronological stream.
    Access decision: accept (p >= tau_acc) or challenge with a strong factor (step-up).
    A genuine user passes the challenge w.p. p_gen_pass (sample becomes *verified*); an attacker
    passes w.p. p_att_pass (default 0) -- failing locks the session and ends the attack burst.
    Update decision: taken per episode of W presented samples by Authenticator.admit(), and applied
    by maybe_update() subject to m_min, privacy budget and pacing."""
    kind, rate = CONDITIONS[cond]
    rng = np.random.default_rng(seed)
    out_rng = np.random.default_rng(seed + 17)               # step-up outcome randomness
    t0 = time.perf_counter()
    A = Authenticator(run['anchor'], run['bg_units'], run['cal_g'], run['cal_bg'], cfg, method, acct, rng)
    A.fit(cfg['init_steps']); t_init = time.perf_counter() - t0
    psr_init = float(np.mean(A.score(run['att_hold']) >= cfg['tau_acc']))
    Xs, ss = run['stream_X'], run['stream_s']; n = len(Xs); horizon = n
    plan = set(attack_plan(n, rate, cfg['burst'], seed + 7919 * sid(run['victim'])))
    apool = run['att_pool_raw']; aptr = 0; alpha = 0.0; vleak = run['leak_raw']
    mmin = cfg['m_min'] if method['silent'] == 'risk' or method['pacing'] else 1
    rec = []; ep = []; upd_t = []; ckpt = []; inf_t = []; tcount = [0]
    def flush():
        if method['adaptive'] and ep:
            X = np.vstack([e[0] for e in ep]); p = np.array([e[1] for e in ep])
            ver = np.array([e[2] for e in ep]); f = np.array([e[3] for e in ep]); idx = [e[4] for e in ep]
            ok = A.admit(X, p, ver)
            if ok.any():
                A.queue(X[ok], f[ok])
                for i in np.where(ok)[0]: rec[idx[i]]['admitted'] = True
            t = time.perf_counter()
            if A.maybe_update(tcount[0], horizon, mmin): upd_t.append(time.perf_counter() - t)
        ep.clear()
    def present(x, is_att, sess):
        t = time.perf_counter(); p = float(A.score(x[None])[0]); inf_t.append(time.perf_counter() - t)
        if p >= cfg['tau_acc']: dec, ver = 'accept', False
        else:
            passed = out_rng.random() < (cfg['p_att_pass'] if is_att else cfg['p_gen_pass'])
            dec, ver = ('stepup_pass' if passed else 'stepup_fail'), passed
        rec.append(dict(p=p, dec=dec, att=is_att, sess=int(sess), admitted=False, verified=ver))
        if ver: A.add_verified(x[None])
        ep.append((x[None], p, ver, is_att, len(rec) - 1)); tcount[0] += 1
        if len(ep) >= cfg['W']: flush()
        return dec
    prev_s = None
    for i in range(n):
        if ss[i] != prev_s:                                   # checkpoint: model state before a new session
            g = A.score(Xs[ss == ss[i]]); im = A.score(run['probe'])
            ckpt.append(dict(sess=int(ss[i]), eer=eer(g, im), frr=float(np.mean(g < cfg['tau_acc'])),
                             far=float(np.mean(im >= cfg['tau_acc'])), eps=A.eps() if method['dp'] else np.nan,
                             n_updates=A.n_updates))
            prev_s = ss[i]
        if i in plan:                                         # attacker obtains the unlocked device: one burst
            outs = []
            for k in range(cfg['burst']):
                if kind == 'zero': xa = apool[aptr % len(apool)]
                elif kind == 'oracle':   # white-box confidence-aware: largest step toward own behaviour that the
                    cand = (1 - ALPHA_GRID[:, None]) * vleak[k % len(vleak)] + ALPHA_GRID[:, None] * apool[aptr % len(apool)]
                    pc = A.score(run['Z'](run['transform'](cand)))       # model still scores >= tau_upd
                    okc = np.where(pc >= cfg['tau_upd'])[0]; xa = cand[okc.max()] if len(okc) else cand[0]
                else: xa = (1 - alpha) * vleak[k % len(vleak)] + alpha * apool[aptr % len(apool)]
                aptr += 1
                o = present(run['Z'](run['transform'](xa[None]))[0], True, ss[i]); outs.append(o)
                if o == 'stepup_fail': break                  # locked out: burst ends
            if kind == 'adaptive':                            # decision-feedback hill climbing (grey box)
                acc = np.mean([o == 'accept' for o in outs]) if len(outs) == cfg['burst'] else 0.0
                alpha = min(1.0, alpha + cfg['adapt_step_up']) if acc >= 0.8 else max(0.0, alpha - cfg['adapt_step_down'])
        present(Xs[i], False, ss[i])
    flush()
    R = pd.DataFrame(rec); G = R[~R.att]; Aa = R[R.att]
    late = G.sess >= (G.sess.max() - 2)
    delays = []
    if len(Aa):
        idx = np.where(R.att.values)[0]; starts = [idx[0]] + [j for a_, j in zip(idx[:-1], idx[1:]) if j != a_ + 1]
        for st in starts:
            d = 0
            while st + d < len(R) and R.att.values[st + d] and R.dec.values[st + d] == 'accept': d += 1
            delays.append(d)
    ck = pd.DataFrame(ckpt)
    out = dict(
        online_FRR=float(np.mean(G.dec != 'accept')), online_FRR_late=float(np.mean(G[late].dec != 'accept')),
        genuine_stepup_rate=float(np.mean(G.dec != 'accept')),
        attack_accept_rate=float(np.mean(Aa.dec == 'accept')) if len(Aa) else np.nan,
        attack_samples_accepted=int((Aa.dec == 'accept').sum()) if len(Aa) else 0,
        detection_delay=float(np.mean(delays)) if delays else np.nan,
        n_updates=A.n_updates, n_admitted=int(R.admitted.sum()),
        n_admitted_attack=int(Aa.admitted.sum()) if len(Aa) else 0,
        n_admitted_verified=int(G[G.admitted].verified.sum()) if len(G) else 0,
        buffer_contam_final=float(A.bufflag.mean()) if len(A.bufflag) else 0.0,
        PSR_init=psr_init, PSR_final=float(np.mean(A.score(run['att_hold']) >= cfg['tau_acc'])),
        eps_final=A.eps() if method['dp'] else np.nan,
        eps_final_pld=acct.eps(A.steps, 'pld') if method['dp'] else np.nan,
        ckpt_EER_mean=float(ck.eer.iloc[1:].mean()), ckpt_EER_late=float(ck.eer.iloc[-3:].mean()),
        ckpt_FAR_late=float(ck.far.iloc[-3:].mean()), ckpt_FRR_late=float(ck.frr.iloc[-3:].mean()),
        t_init_s=t_init, t_update_s=float(np.mean(upd_t)) if upd_t else np.nan,
        t_infer_us=float(np.median(inf_t) * 1e6), steps=A.steps,
        max_unit_grad_norm=float(np.max(A.clip_stats)) if A.clip_stats else np.nan,
        n_stream=len(R), n_attack=int(len(Aa)),
    )
    if collect_trace: return out, R, ck, A
    return out, ck

In [ ]:
%%writefile analysis_helpers.py
# ==== analysis helpers: clustered statistics, figures ====
import numpy as np, pandas as pd
from scipy.stats import wilcoxon

def subject_level(df, metric, by=('method', 'cond')):
    """Average repeated runs (seeds) within subject -> one observation per subject (the independent unit)."""
    return df.groupby(list(by) + ['victim'])[metric].mean().reset_index()

def boot_ci_mean(x, n_boot=10000, seed=0, alpha=0.05):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan)
    r = np.random.default_rng(seed); idx = r.integers(0, len(x), (n_boot, len(x)))
    m = x[idx].mean(1); return tuple(np.quantile(m, [alpha / 2, 1 - alpha / 2]))

def summarize(df, metrics, by=('cond', 'method')):
    rows = []
    for key, g in df.groupby(list(by)):
        row = dict(zip(by, key if isinstance(key, tuple) else (key,)))
        for m in metrics:
            s = g.groupby('victim')[m].mean()
            lo, hi = boot_ci_mean(s.values)
            row[m] = s.mean(); row[m + '_lo'] = lo; row[m + '_hi'] = hi
        row['n_subjects'] = g.victim.nunique(); row['n_runs'] = len(g)
        rows.append(row)
    return pd.DataFrame(rows)

def paired_test(a, b, label, alternative='two-sided'):
    """a, b: Series indexed by subject (already averaged over seeds). Tests median(a-b) = 0."""
    idx = a.index.intersection(b.index); d = (a.loc[idx] - b.loc[idx]).astype(float).dropna()
    d = d.round(10)                      # exact ties identified after rounding to 1e-10 (floating-point noise is not a difference)
    lo, hi = boot_ci_mean(d.values)
    nz = d[d != 0]
    if len(nz) < 5: p, stat = np.nan, np.nan
    else:
        res = wilcoxon(nz.values, alternative=alternative, zero_method='wilcox'); p, stat = res.pvalue, res.statistic
    r_pos = (nz > 0).sum(); r_neg = (nz < 0).sum()
    ranks = pd.Series(np.abs(nz.values)).rank().values
    rbc = (ranks[nz.values > 0].sum() - ranks[nz.values < 0].sum()) / ranks.sum() if len(nz) else np.nan
    return dict(comparison=label, n_subjects=len(d), mean_diff=d.mean(), ci_lo=lo, ci_hi=hi,
                median_diff=d.median(), n_pos=int(r_pos), n_neg=int(r_neg), n_zero=int((d == 0).sum()),
                wilcoxon_W=stat, p_value=p, rank_biserial=rbc)

def holm(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    order = np.argsort(p[ok]); m = ok.sum(); adj = np.empty(m); prev = 0
    for k, i in enumerate(order):
        prev = max(prev, min(1.0, (m - k) * p[ok][i])); adj[i] = prev
    out[ok] = adj; return out

In [ ]:
import importlib, pacau_core, analysis_helpers
importlib.reload(pacau_core); importlib.reload(analysis_helpers)
from pacau_core import *; from analysis_helpers import *
print('methods:', list(METHODS)); print('conditions:', list(CONDITIONS))

## 5. Dataset loading and integrity checks

In [ ]:
SOURCES = {
 'cmu_mirrorA': ('https://raw.githubusercontent.com/rehassachdeva/Anomaly-Detection-for-Keystroke-Dynamics/2a43e09645cfd7823f33768617f3343265d05a49/DSL-StrongPasswordData.csv',
                 'b11d23538b1865fa6ecf4e8b78567caa312e9c1027604bb022fcc6ad7eaa7a33', 'DSL-StrongPasswordData.csv'),
 'cmu_mirrorB': ('https://raw.githubusercontent.com/LesCadets/keystroke-dynamics-benchmark-dataset/acf5f5558653a409ea1f3785905fe0c4fdd2bdd1/data/StrongPasswordData.csv',
                 '38612f33475c8f8ca63ba2fe2c8e6a17a683840eb64721c5ad8b5636ec0fa36e', 'StrongPasswordData_mirrorB.csv'),
 'touch_feat': ('https://raw.githubusercontent.com/thalevi/Touchalytics-Deep-Learning/0d52b05db24d169e0a4d9c3bf231e02eddf44081/data/featMat.csv',
                '75da9d7bb6d9b30066b1bf9b36f3cadd7d09a809235d9a36d24a383ed64713f9', 'touchalytics_featMat.csv'),
 'touch_raw':  ('https://raw.githubusercontent.com/thalevi/Touchalytics-Deep-Learning/0d52b05db24d169e0a4d9c3bf231e02eddf44081/data/data.csv',
                '091e5b75f206a0a1706655e0442199c8e66b6643f55762412ae1fa4faf5dce06', 'touchalytics_raw.csv'),
}
OFFICIAL_CMU = 'https://www.cs.cmu.edu/~keystroke/DSL-StrongPasswordData.csv'
import urllib.request
def sha256(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()
download_log = []
for k, (url, h, fn) in SOURCES.items():
    p = os.path.join(DATA, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: raise RuntimeError(f'Download failed for {k}: {e}. Place {fn} in {DATA} manually and re-run.')
    got = sha256(p); download_log.append(dict(key=k, url=url, file=fn, bytes=os.path.getsize(p), sha256=got, expected=h, ok=got == h))
    assert got == h, f'SHA-256 mismatch for {k}'
try:
    urllib.request.urlopen(OFFICIAL_CMU, timeout=10); official = 'reachable'
except Exception as e: official = f'unreachable from this environment ({type(e).__name__})'
pd.DataFrame(download_log).to_csv(os.path.join(OUT, 'download_log.csv'), index=False)
print('Official CMU host:', official); display(pd.DataFrame(download_log)[['key', 'bytes', 'ok']])

In [ ]:
cmu, cmu_feats, cmu_df = load_cmu(os.path.join(DATA, 'DSL-StrongPasswordData.csv'))
mB = pd.read_csv(os.path.join(DATA, 'StrongPasswordData_mirrorB.csv')).sort_values(['subject', 'sessionIndex', 'rep']).reset_index(drop=True)
num = cmu_df.select_dtypes('number').columns
assert (cmu_df.subject.values == mB.subject.values).all() and np.abs(cmu_df[num].values - mB[num].values).max() == 0
audit = dict(rows=len(cmu_df), subjects=cmu_df.subject.nunique(), sessions=sorted(cmu_df.sessionIndex.unique().tolist()),
             reps_per_session=int(cmu_df.groupby(['subject', 'sessionIndex']).size().unique()[0]),
             missing=int(cmu_df.isna().sum().sum()), duplicate_rows=int(cmu_df.duplicated().sum()),
             duplicate_feature_vectors=int(cmu_df[cmu_feats].duplicated().sum()), n_features=len(cmu_feats),
             negative_UD_values=int((cmu_df[[c for c in cmu_feats if c.startswith('UD')]] < 0).sum().sum()),
             mirrors_identical=True)
assert audit['subjects'] == 51 and audit['sessions'] == list(range(1, 9)) and audit['reps_per_session'] == 50
print(json.dumps(audit, indent=1))
# Replication sanity check: Killourhy & Maxion (2009) scaled-Manhattan detector, their protocol (train reps 1-200, test 201-400,
# impostors = first 5 reps of every other subject). Published average EER: 0.096.
eers = []
for s in cmu:
    X = cmu[s]['X_raw']; tr, te = X[:200], X[200:]
    mu, mad = tr.mean(0), np.abs(tr - tr.mean(0)).mean(0)
    sc = lambda Z: -np.sum(np.abs(Z - mu) / mad, 1)
    imp = np.vstack([cmu[o]['X_raw'][:5] for o in cmu if o != s])
    eers.append(eer(sc(te), sc(imp)))
print(f'Scaled-Manhattan replication EER = {np.mean(eers):.3f} (published 0.096)')
audit['scaled_manhattan_replication_EER'] = float(np.mean(eers))

In [ ]:
touch, touch_feats, touch_meta, touch_dropped = load_touch(os.path.join(DATA, 'touchalytics_featMat.csv'), os.path.join(DATA, 'touchalytics_raw.csv'))
followup = touch_meta.groupby('user').days_from_first.max()
touch_victims = sorted(followup[followup >= 1].index.tolist())
touch_audit = dict(users=len(touch), features=len(touch_feats), stroke_rows_dropped_nonnumeric_or_nan=touch_dropped,
                   window_strokes=5, users_with_followup_ge_1day=len(touch_victims),
                   followup_days_min=float(followup[touch_victims].min()), followup_days_max=float(followup[touch_victims].max()),
                   windows_per_user_median=float(np.median([len(v['sess']) for v in touch.values()])))
print(json.dumps(touch_audit, indent=1))
json.dump(dict(cmu=audit, touchalytics=touch_audit), open(os.path.join(OUT, 'data_audit.json'), 'w'), indent=1)

## 6. Exploratory data analysis: is there drift to adapt to?

In [ ]:
tot = cmu_df.assign(total=cmu_df[[c for c in cmu_feats if c.startswith('H.') or c.startswith('UD.')]].sum(axis=1))
per = tot.groupby(['subject', 'sessionIndex']).total.median().unstack()
rel = per.div(per[1], axis=0)
# displacement of each session mean from session 1, in units of session-1 within-subject SD (transformed space)
disp = []
for s in cmu:
    Xt = cmu_transform(cmu[s]['X_raw']); ss = cmu[s]['sess']; m1, sd1 = Xt[ss == 1].mean(0), Xt[ss == 1].std(0) + 1e-6
    disp.append([np.sqrt(np.mean(((Xt[ss == k].mean(0) - m1) / sd1) ** 2)) for k in range(1, 9)])
disp = np.array(disp)
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
ax[0].plot(range(1, 9), rel.median(), 'o-'); ax[0].fill_between(range(1, 9), rel.quantile(.25), rel.quantile(.75), alpha=.25)
ax[0].set(xlabel='Session', ylabel='Typing time rel. to session 1', title='(a) Practice effect (median, IQR)')
ax[1].plot(range(1, 9), np.median(disp, 0), 'o-', c='C3'); ax[1].fill_between(range(1, 9), np.quantile(disp, .25, 0), np.quantile(disp, .75, 0), color='C3', alpha=.25)
ax[1].set(xlabel='Session', ylabel='Mean shift / session-1 SD', title='(b) Feature-space displacement')
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig1_drift_eda.png')); plt.show()
print('Median session-8 typing time relative to session 1:', round(float(rel[8].median()), 3))
print('Wilcoxon session 8 vs 1 typing time:', scipy.stats.wilcoxon(per[8], per[1]))

## 7. Pre-processing and leakage-safe splitting

* **Feature transform** is fixed *a priori* (signed log of timings in 10 ms units; signed `log1p` for touch) — no statistics are fitted on private data, which keeps the DP guarantee clean.
* **Standardisation and score calibration** use a *public reference set* (`bg_cal` subjects) that is disjoint from the background subjects whose data train the model (`bg_train`), from the attackers/probes and from the victims.
* **Per seed**, the 51 subjects are randomly partitioned into 14 `bg_train`, 6 `bg_cal`, 11 attackers/probe impostors and 20 victims. Each victim gets one designated attacker; probes for threshold-free EER come from the *other* attackers.
* **Victim timeline:** session-1 repetitions 1–40 = enrolment (training anchor), 41–50 = calibration hold-out, sessions 2–8 (350 samples) = chronological stream. Nothing is shuffled.
* **Attacker data:** the first 70% of the attacker's samples feed injections; the last 30% are held out and used only to measure poisoning success (PSR).

In [ ]:
for sd in SEEDS_MAIN:
    P_ = partition(list(cmu), 14, 6, 11, seed=sd); sets = [set(v) for v in P_.values()]
    assert sum(len(s) for s in sets) == 51 and len(set().union(*sets)) == 51, 'roles overlap'
P0 = partition(list(cmu), 14, 6, 11, seed=0); print({k: len(v) for k, v in P0.items()})
v0 = P0['victims'][0]; run0 = build_run(cmu, cmu_transform, P0, v0, P0['attackers'][sid(v0) % 11], DEFAULT_CFG, seed=0)
assert run0['stream_s'].min() == 2 and (np.diff(run0['stream_s']) >= 0).all(), 'stream must be chronological, session>=2'
print('victim', v0, '| anchor', run0['anchor'].shape, '| cal', run0['cal_g'].shape, '| stream', run0['stream_X'].shape,
      '| bg units', len(run0['bg_units']), '| probes', run0['probe'].shape, '| attacker hold-out', run0['att_hold'].shape)

## 8. Sequential authentication stream construction

Prequential (*test-then-train*) protocol: every sample is scored by the model as it stands **before** that sample can influence any update. Attacker bursts of 10 samples are *inserted* (not substituted) at random stream positions, so the genuine stream is identical across conditions and methods, enabling paired comparisons. At each session boundary the current model is evaluated on the next session's genuine samples versus probe impostors (threshold-free EER) — those samples have not been seen by the model at that point.

**Threat model.**
* *Attacker:* an insider (family member, colleague, thief) who gains physical use of the victim's already-unlocked device during a session (the post-login hijacking scenario in the proposal's problem statement). Goal: make the adaptive model accept his own natural behaviour later (a persistent backdoor).
* *Capabilities:* C1/C2 zero-effort — types naturally (10% / 30% of stream positions). C3 white-box confidence-aware — knows the model and update threshold (score oracle), holds 5 leaked victim samples, and synthesises each injected timing vector as the largest step from a leaked victim sample toward his own behaviour that still scores ≥ τ_upd. C4 grey-box — observes only accept/challenge outcomes and hill-climbs. The attacker does **not** know the novelty statistic and **cannot pass the strong factor** (assumption A1; `p_att_pass` is configurable).
* *Out of scope:* compromise of the server or of the strong factor, presentation attacks on the strong factor, robotic replay at scale.

In [ ]:
print(pd.Series({c: CONDITIONS[c] for c in CONDITIONS}, name='(kind, rate)'))
plan = attack_plan(len(run0['stream_X']), 0.20, DEFAULT_CFG['burst'], 123)
print('example burst insertion positions (C3, 20%):', plan)

## 9. Baseline model implementation

All methods share one verifier (logistic regression on 31 timing features), one optimiser (Poisson-sampled mini-batch SGD, lr = 0.05, L2 = 1e-3, 150 initial steps, 25 steps per update), one buffer policy (enrolment anchor always retained + FIFO of 100 admitted samples) and one access rule. They differ only in the factors under study:

| Method | DP | Silent (unlabelled) admissions | Step-up-verified admissions | Budget / pacing | Role |
|---|---|---|---|---|---|
| B0_static | – | – | – | – | non-adaptive baseline |
| B0_static_DP | ✓ | – | – | – | privacy cost without adaptation |
| B1_selfupdate | – | all accepted | ✓ | – | **B1** conventional self-update |
| B1b_double_thr | – | p ≥ τ_upd | ✓ | – | established double-threshold update (Giot/Mhenni-style) |
| B1c_labels_only | – | none | ✓ | – | supervised update only (Windows-Hello-style) |
| B2_DP_selfupdate | ✓ | all accepted | ✓ | budget, greedy | **B2** DP without gate |
| B2c_DP_labels_only | ✓ | none | ✓ | budget + pacing | DP supervised-only |
| B3_gate_nonDP | – | risk gate | ✓ | – | **B3** gate without DP |
| **P_gate_DP_budget** | ✓ | risk gate | ✓ | budget + pacing | **proposed** |
| A_P_no_* | ablations of P (budget, pacing, novelty, evidence, calibration removed one at a time) | | | | |

The *static* model *is* a supervised classifier baseline; deeper temporal models were deliberately not used: with 40 enrolment samples per user a recurrent model is not identifiable, and the study's object is the update policy, not the matcher. Secondary modality/fusion ablation is **not possible**: both datasets are single-modality, and combining unrelated datasets would fabricate multimodal identities.

## 10. Privacy mechanism implementation and verification

**Mechanism.** Unit-level DP-SGD (Abadi et al. 2016; user-level variant as in McMahan et al. 2018): Poisson sampling of units with rate q = 0.2, per-unit mean gradient clipped to L2 norm C = 1, Gaussian noise N(0, σ²C²I) with σ = 3, accounting with the RDP accountant of `dp_accounting` (PLD as a tighter cross-check), δ = 1e-5.

**Protected units and adjacency (add/remove one unit).**
* *Background enrollees (`bg_train`)* — **participant-level** guarantee: all of a participant's samples form one unit. Their data influence the system only through DP-SGD steps; the update gate, novelty statistic and calibration never touch them, so the guarantee covers the **entire sequence of released weight vectors** (adaptive composition over all steps, initial + updates).
* *The user's own samples* — each sample is a unit inside DP-SGD, **but** admission decisions, score calibration (enrolment hold-out + recent verified samples) and the novelty anchor are computed non-privately on the user's data. We therefore make **no formal privacy claim for the user's own behaviour**; record-level protection holds only conditional on the admitted set.
* *Excluded from the guarantee:* the public reference set `bg_cal` (used for standardisation and calibration negatives), the calibration map, and the gate's decision log.

Because a per-user model has few units (14 background + 40–140 own samples), meaningful ε requires heavy noise; this is a structural limitation, not a tuning failure.

In [ ]:
acct = Accountant(DEFAULT_CFG['q'], DEFAULT_CFG['sigma'], DEFAULT_CFG['delta'])
# (1) clipping bound: every clipped per-unit gradient has norm <= C
r = np.random.default_rng(0); d = 32; n_units = 60
Xb = add_bias(r.normal(size=(600, d - 1))); y = r.integers(0, 2, 600).astype(float); unit = np.repeat(np.arange(n_units), 10)
w = r.normal(size=d) * 3
g_i = (sigmoid(Xb @ w) - y)[:, None] * Xb; G = np.zeros((n_units, d)); np.add.at(G, unit, g_i); G /= 10
Gc = G * np.minimum(1, 1 / np.linalg.norm(G, axis=1))[:, None]
assert np.linalg.norm(Gc, axis=1).max() <= 1 + 1e-9; print('max pre-clip unit norm %.2f -> post-clip %.3f' % (np.linalg.norm(G, axis=1).max(), np.linalg.norm(Gc, axis=1).max()))
# (2) noise calibration: with zero data gradient the update equals lr * N(0, (sigma*C)^2) / (q*n_units)
outs = []
for k in range(4000):
    w1 = sgd_train(np.zeros(d), Xb, np.full(600, 0.5), unit, n_units, 1, 1.0, np.random.default_rng(k), 0.2, clip=1.0, sigma=3.0, l2=0.0)
    outs.append(w1)
emp = np.std(outs, 0).mean(); theo = 3.0 / (0.2 * n_units)
print(f'noise std per coordinate: empirical {emp:.4f} vs theoretical {theo:.4f}'); assert abs(emp / theo - 1) < 0.05
# (3) Poisson sampling rate
sel = (np.random.default_rng(1).random((5000, n_units)) < 0.2).mean(); print('empirical sampling rate', round(sel, 4))
# (4) accountant: monotone in steps, PLD <= RDP
tab = pd.DataFrame([dict(steps=s, eps_rdp=acct.eps(s), eps_pld=acct.eps(s, 'pld')) for s in [150, 175, 200, 250, 300, 400, 500, 650]])
assert tab.eps_rdp.is_monotonic_increasing and (tab.eps_pld <= tab.eps_rdp + 1e-6).all()
A_tmp = Authenticator(run0['anchor'], run0['bg_units'], run0['cal_g'], run0['cal_bg'], DEFAULT_CFG, METHODS['P_gate_DP_budget'], acct, np.random.default_rng(0))
A_tmp.steps = DEFAULT_CFG['init_steps']
print(f"initial training eps = {acct.eps(150):.2f} (RDP) / {acct.eps(150,'pld'):.2f} (PLD) at delta=1e-5; updates affordable within eps<=8: {A_tmp.n_affordable()}")
tab.to_csv(os.path.join(OUT, 'privacy_accounting_table.csv'), index=False); display(tab.round(3))
# group-privacy caveat for record-level reading of a background participant's 50 samples is unnecessary: they form one unit.

## 11. Adaptive update policy implementation

Implemented in `Authenticator.admit()` (eligibility), `queue()` and `maybe_update()` (budget, pacing, m_min) and `run_stream()` (access decisions, step-up outcomes, lock-out). Key design property: **unlabelled admissions are confined to a fixed region around the enrolment anchor** — the trust radius R is the 95th percentile of enrolment novelty and is never recomputed from admitted data — so no number of poisoned-but-admitted samples can move the admissible region; labelled (strong-factor-verified) samples are what let the model follow real drift.

Pre-registered constants (not tuned on evaluation data): τ_acc = 0.5, τ_upd = 0.8, ρ = 0.8, W = 10, m_min = 5, trust quantile 0.95, ε budget 8 (δ = 1e-5), buffer 100, recalibration with the 20 most recent verified samples. The learning rate (0.05) was the only quantity chosen on the pilot partition, by static-model EER only.

In [ ]:
prereg = dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(), config=DEFAULT_CFG, methods=METHODS,
              conditions={k: list(v) for k, v in CONDITIONS.items()}, seeds_main=SEEDS_MAIN, seeds_sensitivity=SEEDS_SENS,
              pilot_seed_excluded=PILOT_SEED, partition=dict(bg_train=14, bg_cal=6, attackers=11, victims=20),
              primary=dict(endpoint='attack_gain = PSR_final(C3_confaware_oracle_20) - PSR_final(C0_drift_only), per victim & seed',
                           comparison='P_gate_DP_budget vs B1_selfupdate', test='two-sided Wilcoxon signed-rank over subjects (seed-averaged)',
                           alpha=0.05, ci='95% subject-cluster bootstrap'),
              secondary_family_holm=['H1a B0_static_DP vs B0_static ckpt_EER_late C0', 'H1b P vs B3 ckpt_EER_late C0',
                                     'H3 P vs B0_static online_FRR C0', 'H4a B3 vs B1 gain', 'H4b P vs B2 gain', 'H4c P vs B3 gain',
                                     'H4d interaction (B2-P)-(B1-B3) gain', 'H5a P vs B2c online_FRR C0', 'H5b P vs B2c gain',
                                     'H5c B3 vs B1c online_FRR C0', 'H5d B3 vs B1c gain'],
              practical_margin_pp=5)
PRE = os.path.join(OUT, 'preregistration.json')
if os.path.exists(PRE):   # never overwrite: compare the frozen protocol with the current code instead
    old = json.load(open(PRE)); same = json.dumps({k: old[k] for k in old if k != 'frozen_at'}, sort_keys=True, default=str) == json.dumps({k: json.loads(json.dumps(prereg, default=str))[k] for k in prereg if k != 'frozen_at'}, sort_keys=True, default=str)
    print('existing pre-registration kept; frozen_at =', old['frozen_at'], '| protocol identical to current code:', same); assert same
else:
    json.dump(prereg, open(PRE, 'w'), indent=1, default=str); print('pre-registration written')

## 12. Adversarial contamination simulation — illustrative trace

In [ ]:
v = P0['victims'][1]; att = P0['attackers'][sid(v) % 11]; runv = build_run(cmu, cmu_transform, P0, v, att, DEFAULT_CFG, seed=0)
fig, ax = plt.subplots(1, 3, figsize=(9, 2.6), sharey=True)
for a_, mn in zip(ax, ['B1_selfupdate', 'B3_gate_nonDP', 'P_gate_DP_budget']):
    out, Rr, ck, A_ = run_stream(runv, DEFAULT_CFG, METHODS[mn], acct, 'C3_confaware_oracle_20', seed=0, collect_trace=True)
    t = np.arange(len(Rr))
    a_.scatter(t[~Rr.att], Rr.p[~Rr.att], s=3, c='C0', label='genuine'); a_.scatter(t[Rr.att], Rr.p[Rr.att], s=6, c='C3', label='attacker')
    a_.scatter(t[Rr.att & Rr.admitted], Rr.p[Rr.att & Rr.admitted], s=22, facecolors='none', edgecolors='k', label='attacker admitted')
    a_.axhline(DEFAULT_CFG['tau_acc'], ls='--', c='grey', lw=.8); a_.axhline(DEFAULT_CFG['tau_upd'], ls=':', c='grey', lw=.8)
    a_.set(title=f"{mn}\nadmitted attack={out['n_admitted_attack']}  PSR {out['PSR_init']:.2f}->{out['PSR_final']:.2f}", xlabel='stream position')
ax[0].set_ylabel('calibrated score p'); ax[0].legend(fontsize=6, loc='lower left'); plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig2_attack_trace.png')); plt.show()

## 13. Main comparative experiments (5 seeds × 20 victims × 14 methods × 5 conditions)

In [ ]:
MAIN_CSV, CK_CSV = os.path.join(OUT, 'results_per_run_main.csv'), os.path.join(OUT, 'checkpoints_main.csv')
def run_grid(data, transform, seeds, methods, conds, cfg, part_sizes, tag, victims_fixed=None):
    rows, cks = [], []; t0 = time.time()
    for sd in seeds:
        if victims_fixed is None: part = partition(list(data), *part_sizes, seed=sd)
        else:
            others = [s for s in data if s not in victims_fixed]; part = partition(others, *part_sizes, seed=sd); part['victims'] = list(victims_fixed)
        acc_ = Accountant(cfg['q'], cfg['sigma'], cfg['delta'])
        for v in part['victims'][:MAX_VICTIMS]:
            att = part['attackers'][sid(v) % len(part['attackers'])]
            run = build_run(data, transform, part, v, att, cfg, seed=sd)
            for cond in conds:
                for mn in methods:
                    out, ck = run_stream(run, cfg, METHODS[mn], acc_, cond, seed=sd)
                    rows.append(dict(tag=tag, seed=sd, victim=str(v), attacker=str(att), cond=cond, method=mn, **out))
                    cks.append(ck.assign(tag=tag, seed=sd, victim=str(v), cond=cond, method=mn))
        print(f'[{tag}] seed {sd} done, {len(rows)} runs, {time.time() - t0:.0f}s', flush=True)
    return pd.DataFrame(rows), pd.concat(cks, ignore_index=True)
if os.path.exists(MAIN_CSV) and not FORCE_RERUN:
    res, ckres = pd.read_csv(MAIN_CSV, dtype={'victim': str}), pd.read_csv(CK_CSV, dtype={'victim': str}); print('loaded cached main results')
else:
    res, ckres = run_grid(cmu, cmu_transform, SEEDS_MAIN, list(METHODS), list(CONDITIONS), DEFAULT_CFG, (14, 6, 11), 'cmu_main')
    res.to_csv(MAIN_CSV, index=False); ckres.to_csv(CK_CSV, index=False)
print(res.shape, res.victim.nunique(), 'distinct victim subjects')

## 14. Ablation and sensitivity analysis

In [ ]:
SENS_CSV = os.path.join(OUT, 'results_sensitivity.csv')
variants = [('eps_budget', 4.0), ('eps_budget', 16.0), ('trust_q', 0.90), ('trust_q', 0.99), ('tau_upd', 0.7), ('tau_upd', 0.9),
            ('W', 5), ('W', 20), ('sigma', 2.0), ('sigma', 4.0), ('m_min', 1), ('m_min', 10), ('recal_verified', 0), ('p_att_pass', 0.1)]
if SMOKE: variants = variants[:2]
if os.path.exists(SENS_CSV) and not FORCE_RERUN: sens = pd.read_csv(SENS_CSV, dtype={'victim': str})
else:
    parts = []
    for k, val in variants:
        cfg = dict(DEFAULT_CFG, **{k: val})
        r_, _ = run_grid(cmu, cmu_transform, SEEDS_SENS, ['P_gate_DP_budget', 'B3_gate_nonDP', 'B1_selfupdate'], ['C0_drift_only', 'C3_confaware_oracle_20'], cfg, (14, 6, 11), f'{k}={val}')
        parts.append(r_.assign(param=k, value=val))
    sens = pd.concat(parts, ignore_index=True); sens.to_csv(SENS_CSV, index=False)
# selective-authentication trade-off: sweep the access threshold
TAU_CSV = os.path.join(OUT, 'results_tau_sweep.csv')
if os.path.exists(TAU_CSV) and not FORCE_RERUN: tau = pd.read_csv(TAU_CSV, dtype={'victim': str})
else:
    parts = []
    for ta in ([0.4, 0.6] if SMOKE else [0.3, 0.4, 0.5, 0.6, 0.7]):
        cfg = dict(DEFAULT_CFG, tau_acc=ta, tau_upd=max(DEFAULT_CFG['tau_upd'], ta + 0.1))
        r_, _ = run_grid(cmu, cmu_transform, [0], ['B0_static', 'B1_selfupdate', 'B1c_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget'], ['C0_drift_only', 'C3_confaware_oracle_20'], cfg, (14, 6, 11), f'tau={ta}')
        parts.append(r_.assign(tau_acc=ta))
    tau = pd.concat(parts, ignore_index=True); tau.to_csv(TAU_CSV, index=False)
print(sens.shape, tau.shape)

## 15. Authentication, privacy, robustness and efficiency evaluation

### 15a. Empirical privacy: membership inference against static models

*Participant-level* attack on background enrollees: for each victim model, a member score is the mean predicted genuine-probability over a `bg_train` subject's training samples; non-members are attacker/probe subjects (never in training). AUC > 0.5 indicates leakage. *Record-level* attack on the user's own enrolment: training samples (session 1, reps 1–40) vs. same-session hold-out (reps 41–50), using the model's loss. A failed attack is **not** evidence of privacy; it only bounds what this specific attack can extract.

In [ ]:
MIA_CSV = os.path.join(OUT, 'results_mia.csv')
if os.path.exists(MIA_CSV) and not FORCE_RERUN: mia = pd.read_csv(MIA_CSV)
else:
    rows = []
    for sd in SEEDS_MAIN:
        part = partition(list(cmu), 14, 6, 11, seed=sd)
        for v in part['victims'][:MAX_VICTIMS]:
            run = build_run(cmu, cmu_transform, part, v, part['attackers'][sid(v) % 11], DEFAULT_CFG, seed=sd)
            nonmem = [run['Z'](cmu_transform(cmu[a]['X_raw']))[np.random.default_rng(sd).choice(400, 50, replace=False)] for a in part['attackers']]
            for mn in ['B0_static', 'B0_static_DP']:
                A_ = Authenticator(run['anchor'], run['bg_units'], run['cal_g'], run['cal_bg'], DEFAULT_CFG, METHODS[mn], acct, np.random.default_rng(sd))
                A_.fit(DEFAULT_CFG['init_steps'])
                # members are trained as negatives -> lower genuine prob; attack score = -mean prob
                mem_s = [-A_.score(U).mean() for U in run['bg_units']]; non_s = [-A_.score(U).mean() for U in nonmem]
                pg = lambda X: sigmoid(add_bias(X) @ A_.w)
                loss_in = -np.log(pg(run['anchor']) + 1e-12); loss_out = -np.log(pg(run['cal_g']) + 1e-12)
                rows.append(dict(seed=sd, victim=str(v), method=mn, auc_participant=auc(np.array(mem_s), np.array(non_s)),
                                 auc_record=auc(-loss_in, -loss_out)))
    mia = pd.DataFrame(rows); mia.to_csv(MIA_CSV, index=False)
display(mia.groupby('method')[['auc_participant', 'auc_record']].agg(['mean', 'std']).round(3))

### 15b. External validation on Touchalytics (touch modality, inter-week follow-up)

In [ ]:
TOUCH_CSV = os.path.join(OUT, 'results_touchalytics.csv')
TCFG = dict(DEFAULT_CFG, n_enroll=30, n_cal=10, burst=5, probe_per_subject=10)
others = [u for u in touch if u not in touch_victims]
tsizes = (10, 5, len(others) - 15)
if os.path.exists(TOUCH_CSV) and not FORCE_RERUN: tres = pd.read_csv(TOUCH_CSV, dtype={'victim': str})
else:
    tres, _ = run_grid(touch, touch_transform, SEEDS_MAIN, ['B0_static', 'B0_static_DP', 'B1_selfupdate', 'B1c_labels_only', 'B2_DP_selfupdate', 'B3_gate_nonDP', 'P_gate_DP_budget'],
                       ['C0_drift_only', 'C2_zero_effort_30', 'C3_confaware_oracle_20'], TCFG, tsizes, 'touch', victims_fixed=touch_victims)
    tres.to_csv(TOUCH_CSV, index=False)
print(tres.shape, 'victims:', tres.victim.nunique())

### 15c. Operational efficiency

In [ ]:
eff = res[res.cond == 'C0_drift_only'].groupby('method')[['t_init_s', 't_update_s', 't_infer_us', 'n_updates', 'steps']].median()
eff['model_params'] = 32 + 2                                   # 31 weights + bias + 2 Platt parameters
eff['on_device_state_kB'] = (34 + 62 + (40 + 100) * 31) * 8 / 1024   # params + novelty stats + anchor + buffer (float64)
eff.round(4).to_csv(os.path.join(OUT, 'efficiency.csv')); display(eff.round(4))
print('Hardware:', platform.processor() or platform.machine(), '| logical CPUs:', os.cpu_count())

## 16. Statistical analysis

In [ ]:
def S(metric, method, cond, df=res):
    g = df[(df.method == method) & (df.cond == cond)]; return g.groupby('victim')[metric].mean()
def gain(method, df=res, attack='C3_confaware_oracle_20'):
    return S('PSR_final', method, attack, df) - S('PSR_final', method, 'C0_drift_only', df)
primary = paired_test(gain('P_gate_DP_budget'), gain('B1_selfupdate'), 'PRIMARY H2: gain(P) - gain(B1)')
sec = [paired_test(S('ckpt_EER_late', 'B0_static_DP', 'C0_drift_only'), S('ckpt_EER_late', 'B0_static', 'C0_drift_only'), 'H1a EER_late: B0_DP - B0'),
       paired_test(S('ckpt_EER_late', 'P_gate_DP_budget', 'C0_drift_only'), S('ckpt_EER_late', 'B3_gate_nonDP', 'C0_drift_only'), 'H1b EER_late: P - B3'),
       paired_test(S('online_FRR', 'P_gate_DP_budget', 'C0_drift_only'), S('online_FRR', 'B0_static', 'C0_drift_only'), 'H3 challenge rate: P - B0'),
       paired_test(gain('B3_gate_nonDP'), gain('B1_selfupdate'), 'H4a gain: B3 - B1'),
       paired_test(gain('P_gate_DP_budget'), gain('B2_DP_selfupdate'), 'H4b gain: P - B2'),
       paired_test(gain('P_gate_DP_budget'), gain('B3_gate_nonDP'), 'H4c gain: P - B3'),
       paired_test(gain('B2_DP_selfupdate') - gain('P_gate_DP_budget'), gain('B1_selfupdate') - gain('B3_gate_nonDP'), 'H4d interaction: (B2-P) - (B1-B3)'),
       paired_test(S('online_FRR', 'P_gate_DP_budget', 'C0_drift_only'), S('online_FRR', 'B2c_DP_labels_only', 'C0_drift_only'), 'H5a challenge rate: P - B2c'),
       paired_test(gain('P_gate_DP_budget'), gain('B2c_DP_labels_only'), 'H5b gain: P - B2c'),
       paired_test(S('online_FRR', 'B3_gate_nonDP', 'C0_drift_only'), S('online_FRR', 'B1c_labels_only', 'C0_drift_only'), 'H5c challenge rate: B3 - B1c'),
       paired_test(gain('B3_gate_nonDP'), gain('B1c_labels_only'), 'H5d gain: B3 - B1c')]
tests = pd.DataFrame([primary] + sec); tests['p_holm'] = np.r_[primary['p_value'], holm([s['p_value'] for s in sec])]
tests['family'] = ['primary'] + ['secondary (Holm)'] * len(sec)
tests['practically_significant'] = tests[['ci_lo', 'ci_hi']].abs().min(axis=1).where(np.sign(tests.ci_lo) == np.sign(tests.ci_hi), 0) >= 0.05
tests.to_csv(os.path.join(OUT, 'hypothesis_tests.csv'), index=False)
pd.set_option('display.width', 220); display(tests[['comparison', 'n_subjects', 'mean_diff', 'ci_lo', 'ci_hi', 'n_pos', 'n_neg', 'n_zero', 'p_value', 'p_holm', 'rank_biserial', 'practically_significant']].round(4))
# secondary attack conditions (descriptive, not in the confirmatory family)
desc = []
for c in ['C1_zero_effort_10', 'C2_zero_effort_30', 'C4_greybox_feedback_20']:
    for a_, b_ in [('P_gate_DP_budget', 'B1_selfupdate'), ('B3_gate_nonDP', 'B1_selfupdate')]:
        desc.append(paired_test(gain(a_, attack=c), gain(b_, attack=c), f'{c}: gain {a_} - {b_}'))
desc = pd.DataFrame(desc); desc.to_csv(os.path.join(OUT, 'exploratory_tests_other_attacks.csv'), index=False); display(desc.round(4))

## 17. Results tables and publication-quality figures

In [ ]:
METRICS = ['online_FRR', 'online_FRR_late', 'ckpt_EER_late', 'ckpt_FAR_late', 'PSR_init', 'PSR_final', 'attack_accept_rate',
           'detection_delay', 'n_updates', 'n_admitted_attack', 'buffer_contam_final', 'eps_final', 'eps_final_pld']
agg = summarize(res, METRICS); agg.to_csv(os.path.join(OUT, 'aggregate_metrics_ci.csv'), index=False)
gains = []
for c in [c for c in CONDITIONS if c != 'C0_drift_only']:
    for m in METHODS:
        gser = gain(m, attack=c); lo, hi = boot_ci_mean(gser.values); gains.append(dict(cond=c, method=m, attack_gain=gser.mean(), lo=lo, hi=hi, n=len(gser)))
gains = pd.DataFrame(gains); gains.to_csv(os.path.join(OUT, 'attack_gain_ci.csv'), index=False)
main_tab = agg[agg.cond.isin(['C0_drift_only', 'C3_confaware_oracle_20'])][['cond', 'method', 'online_FRR', 'ckpt_EER_late', 'ckpt_FAR_late', 'PSR_final', 'n_updates', 'n_admitted_attack', 'eps_final']]
display(main_tab.round(3)); display(gains[gains.cond == 'C3_confaware_oracle_20'].round(3))

In [ ]:
order = ['B0_static', 'B0_static_DP', 'B1_selfupdate', 'B1b_double_thr', 'B1c_labels_only', 'B2_DP_selfupdate', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget',
         'A_P_no_budget', 'A_P_no_pacing', 'A_P_no_novelty', 'A_P_no_evidence', 'A_P_no_calib']
col = {m: ('C2' if m.startswith('P_') else 'C1' if m.startswith('A_') else 'C0' if m.startswith('B0') else 'C7') for m in order}
# Fig 3: prequential EER per session (C0) for main methods
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.8))
for m, ls in [('B0_static', '-'), ('B0_static_DP', '--'), ('B1_selfupdate', '-'), ('B1c_labels_only', '-.'), ('B3_gate_nonDP', '-'), ('P_gate_DP_budget', '-')]:
    g = ckres[(ckres.method == m) & (ckres.cond == 'C0_drift_only')].groupby(['victim', 'sess']).eer.mean().groupby('sess').mean()
    f = ckres[(ckres.method == m) & (ckres.cond == 'C0_drift_only')].groupby(['victim', 'sess']).frr.mean().groupby('sess').mean()
    ax[0].plot(g.index, g.values, ls, marker='o', ms=3, label=m); ax[1].plot(f.index, f.values, ls, marker='o', ms=3, label=m)
ax[0].set(xlabel='Session (model state at session start)', ylabel='Prequential EER', title='(a) Threshold-free EER, drift only')
ax[1].set(xlabel='Session', ylabel='Genuine rejection at τ_acc', title='(b) Next-session genuine rejection')
ax[1].legend(fontsize=6); plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig3_drift_tracking.png')); plt.show()
# Fig 4: attack gain with CIs
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.2), sharey=True)
for a_, c in zip(ax, ['C2_zero_effort_30', 'C3_confaware_oracle_20']):
    gg = gains[gains.cond == c].set_index('method').loc[order]
    a_.barh(range(len(order)), gg.attack_gain, xerr=[gg.attack_gain - gg.lo, gg.hi - gg.attack_gain], color=[col[m] for m in order], alpha=.85)
    a_.axvline(0, c='k', lw=.6); a_.set(title=c, xlabel='Δ attacker acceptance vs. drift-only (PSR)')
ax[0].set_yticks(range(len(order))); ax[0].set_yticklabels(order, fontsize=7); ax[0].invert_yaxis()
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig4_attack_gain.png')); plt.show()

In [ ]:
# Fig 5: security-availability-privacy trade-off (C3): challenge rate vs final attacker acceptance, eps annotated
a3 = agg[agg.cond == 'C3_confaware_oracle_20'].set_index('method').loc[order]
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3))
ax[0].errorbar(a3.online_FRR, a3.PSR_final, xerr=[a3.online_FRR - a3.online_FRR_lo, a3.online_FRR_hi - a3.online_FRR], yerr=[a3.PSR_final - a3.PSR_final_lo, a3.PSR_final_hi - a3.PSR_final], fmt='none', ecolor='lightgrey')
for m in order:
    ax[0].scatter(a3.loc[m, 'online_FRR'], a3.loc[m, 'PSR_final'], c=col[m], s=18)
    ax[0].annotate(m.replace('_', ' ')[:14] + ('' if np.isnan(a3.loc[m, 'eps_final']) else f" ε={a3.loc[m, 'eps_final']:.1f}"), (a3.loc[m, 'online_FRR'], a3.loc[m, 'PSR_final']), fontsize=5)
ax[0].set(xlabel='Genuine challenge (step-up) rate', ylabel='Final attacker acceptance (PSR)', title='(a) C3: friction vs. security')
for m in ['B1_selfupdate', 'B1c_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']:
    t3 = tau[(tau.method == m) & (tau.cond == 'C3_confaware_oracle_20')].groupby('tau_acc')[['online_FRR', 'PSR_final']].mean()
    ax[1].plot(t3.online_FRR, t3.PSR_final, 'o-', ms=3, label=m)
ax[1].set(xlabel='Genuine challenge rate', ylabel='PSR (C3)', title='(b) Access-threshold sweep τ_acc 0.3–0.7'); ax[1].legend(fontsize=6)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig5_tradeoff.png')); plt.show()
# Fig 6: privacy budget consumption and sensitivity
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.8))
for m in ['B2_DP_selfupdate', 'B2c_DP_labels_only', 'P_gate_DP_budget', 'A_P_no_budget', 'A_P_no_pacing']:
    e = ckres[(ckres.method == m) & (ckres.cond == 'C0_drift_only')].groupby('sess').eps.mean(); ax[0].plot(e.index, e.values, 'o-', ms=3, label=m)
ax[0].axhline(DEFAULT_CFG['eps_budget'], c='k', ls=':', lw=.8); ax[0].set(xlabel='Session', ylabel='cumulative ε (RDP, δ=1e-5)', title='(a) Budget consumption'); ax[0].legend(fontsize=6)
sp = sens[sens.method == 'P_gate_DP_budget']
sg = (sp[sp.cond == 'C3_confaware_oracle_20'].groupby(['param', 'value', 'victim']).PSR_final.mean() - sp[sp.cond == 'C0_drift_only'].groupby(['param', 'value', 'victim']).PSR_final.mean()).groupby(['param', 'value']).mean()
sf = sp[sp.cond == 'C0_drift_only'].groupby(['param', 'value']).online_FRR.mean()
lab = [f'{p}={v:g}' for p, v in sg.index]
ax[1].scatter(sf.loc[sg.index], sg.values, s=12)
for l_, x_, y_ in zip(lab, sf.loc[sg.index], sg.values): ax[1].annotate(l_, (x_, y_), fontsize=5)
base_g = gain('P_gate_DP_budget', df=res[res.seed.isin(SEEDS_SENS)]).mean(); base_f = S('online_FRR', 'P_gate_DP_budget', 'C0_drift_only', res[res.seed.isin(SEEDS_SENS)]).mean()
ax[1].scatter([base_f], [base_g], marker='*', s=60, c='C3'); ax[1].annotate('default', (base_f, base_g), fontsize=6)
ax[1].set(xlabel='Challenge rate (C0)', ylabel='Attack gain (C3)', title='(b) One-at-a-time sensitivity of P')
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig6_privacy_sensitivity.png')); plt.show()
sens_tab = pd.DataFrame({'attack_gain_C3': sg, 'challenge_rate_C0': sf.loc[sg.index]}).reset_index(); sens_tab.to_csv(os.path.join(OUT, 'sensitivity_summary.csv'), index=False)

In [ ]:
# Touchalytics replication table + figure
tg = []
for c in ['C2_zero_effort_30', 'C3_confaware_oracle_20']:
    for m in tres.method.unique():
        gs = S('PSR_final', m, c, tres) - S('PSR_final', m, 'C0_drift_only', tres); lo, hi = boot_ci_mean(gs.values)
        tg.append(dict(cond=c, method=m, attack_gain=gs.mean(), lo=lo, hi=hi))
tg = pd.DataFrame(tg)
tagg = summarize(tres, ['online_FRR', 'ckpt_EER_late', 'ckpt_FAR_late', 'PSR_final', 'n_admitted_attack', 'eps_final'])
tagg.to_csv(os.path.join(OUT, 'touchalytics_aggregate_ci.csv'), index=False); tg.to_csv(os.path.join(OUT, 'touchalytics_attack_gain_ci.csv'), index=False)
display(tagg[tagg.cond == 'C0_drift_only'][['method', 'online_FRR', 'ckpt_EER_late', 'ckpt_FAR_late', 'eps_final', 'n_subjects']].round(3)); display(tg.round(3))
ttests = pd.DataFrame([paired_test(gain('P_gate_DP_budget', tres), gain('B1_selfupdate', tres), 'Touch: gain(P) - gain(B1) [C3]'),
                       paired_test(S('online_FRR', 'P_gate_DP_budget', 'C0_drift_only', tres), S('online_FRR', 'B0_static', 'C0_drift_only', tres), 'Touch: challenge P - B0 [C0]')])
ttests.to_csv(os.path.join(OUT, 'touchalytics_tests.csv'), index=False); display(ttests.round(4))
fig, ax = plt.subplots(figsize=(4.2, 2.6)); tt = tg[tg.cond == 'C3_confaware_oracle_20'].set_index('method')
ax.barh(tt.index, tt.attack_gain, xerr=[tt.attack_gain - tt.lo, tt.hi - tt.attack_gain], color='C7'); ax.axvline(0, c='k', lw=.6)
ax.set(xlabel='Attack gain (PSR, C3)', title=f'Touchalytics external validation (n={tres.victim.nunique()} users)'); plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig7_touchalytics.png')); plt.show()
# privacy MIA figure
fig, ax = plt.subplots(figsize=(3.6, 2.6)); mm = mia.groupby(['method', 'victim'])[['auc_participant', 'auc_record']].mean().groupby('method')
mu_, lo_, hi_ = mm.mean(), mm.agg(lambda s: boot_ci_mean(s.values)[0]), mm.agg(lambda s: boot_ci_mean(s.values)[1])
x = np.arange(2)
for i, c in enumerate(['auc_participant', 'auc_record']):
    ax.errorbar(x + i * .15, mu_[c], yerr=[mu_[c] - lo_[c], hi_[c] - mu_[c]], fmt='o', label=c)
ax.axhline(.5, c='k', ls=':', lw=.8); ax.set_xticks(x + .07); ax.set_xticklabels(mu_.index, fontsize=7); ax.set(ylabel='MIA AUC', title='Membership inference'); ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'fig8_mia.png')); plt.show()
mia_tests = pd.DataFrame([paired_test(mia[mia.method == 'B0_static_DP'].groupby('victim')[c].mean(), mia[mia.method == 'B0_static'].groupby('victim')[c].mean(), f'MIA {c}: DP - nonDP') for c in ['auc_participant', 'auc_record']])
mia_tests.to_csv(os.path.join(OUT, 'mia_tests.csv'), index=False); display(mia_tests.round(4))

## 18. Hypothesis verdicts (computed), limitations and threats to validity

In [ ]:
# expected sign of mean_diff under each pre-registered hypothesis (-1 lower, +1 higher, 0 two-sided/no direction)
EXPECTED = [-1, +1, +1, -1, -1, -1, -1, 0, -1, -1, -1, -1]
def verdict(row, exp):
    if np.isnan(row['p_holm']): return 'not testable'
    if row['p_holm'] >= 0.05: return 'not supported (no significant difference)'
    margin = ' (practically significant)' if row['practically_significant'] else ' (below 5 pp practical margin)'
    if exp == 0: return 'difference detected' + margin
    return ('supported' if np.sign(row['mean_diff']) == exp else 'contradicted (significant, opposite direction)') + margin
tests['expected_sign'] = EXPECTED
tests['verdict'] = [verdict(r, e) for (_, r), e in zip(tests.iterrows(), EXPECTED)]
display(tests[['comparison', 'expected_sign', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'verdict']].round(4))
tests.to_csv(os.path.join(OUT, 'hypothesis_tests.csv'), index=False)
# Reviewer 1, point 2: how to read "supported"/"contradicted" relative to the 5-pp practical margin
def practical_reading(lo, hi, m=0.05):
    if lo > 0 or hi < 0:
        if min(abs(lo), abs(hi)) >= m: return 'practically significant (whole CI beyond 5 pp)'
        if max(abs(lo), abs(hi)) < m: return 'detectable but negligible (whole CI within 5 pp)'
        return 'detectable; size uncertain relative to 5 pp'
    return 'no meaningful difference (CI within +-5 pp)' if (lo > -m and hi < m) else 'inconclusive (CI spans 0 and 5 pp)'
tests['practical_reading'] = [practical_reading(r.ci_lo, r.ci_hi) for r in tests.itertuples()]
display(tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'verdict', 'practical_reading']].round(4))
tests.to_csv(os.path.join(OUT, 'hypothesis_tests.csv'), index=False)


**Limitations and threats to validity (independent of the numerical outcome).**

1. *External validity.* CMU is 51 desktop typists of one English password (2009); Touchalytics is 41 users on four 2012 Android phones. Neither reflects Amharic input, Ethiopian users, smartphone keystroke dynamics or Fayda/MOSIP operating conditions. This is a simulation on public data, **not** an evaluation of Fayda or MOSIP.
2. *Drift.* CMU drift is mainly a practice effect over 8 sessions ≥1 day apart; Touchalytics follow-up covers only the users who returned (n reported above) after about a week. Months-scale drift, device changes and context changes are not represented.
3. *Fixed-text proxy.* Each CMU repetition is treated as one continuous-authentication observation; real CA uses free-text windows.
4. *Synthetic attacks.* C3/C4 are controlled simulations; synthesised timing vectors assume an attacker who can reproduce arbitrary timings (e.g., with software injection). Results are not evidence of observed real-world attacks. Assumption A1 (attacker fails the strong factor) is optimistic; `p_att_pass` is varied in the sensitivity analysis.
5. *Privacy scope.* The formal guarantee is participant-level for background enrollees with respect to released weights only. The user's own behaviour, the calibration map and the gate's decision log are not protected. Small unit counts make ε large relative to population-scale DP deployments.
6. *Pre-registration timestamp.* The protocol file was first written (about 11:50 UTC, before the main grid finished at 12:17) and then overwritten with a later timestamp when the notebook was resumed from cache; its content is code-identical. The notebook now refuses to overwrite it.
7. *Pilot leakage.* Learning rate, Platt smoothing, verified-label recalibration and the C3 attack design were fixed after a pilot on partition seed 999, which uses the same 51 subjects in different roles.
8. *Statistics.* Subjects are the independent unit (seed-averaged); victims overlap across seeds with different background/attacker roles, so seeds are not independent replications. Simulated step-up friction is not observed human behaviour.
9. *Model class.* A linear verifier was chosen for identifiability and DP tractability; conclusions about the update policy may not transfer unchanged to deep embedding models.

## 19. Reproducibility manifest and references

In [ ]:
manifest = dict(generated_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), versions=versions, downloads=download_log,
                official_cmu_host=official, data_audit=dict(cmu=audit, touchalytics=touch_audit), config=DEFAULT_CFG,
                seeds=dict(main=SEEDS_MAIN, sensitivity=SEEDS_SENS, pilot_excluded=PILOT_SEED),
                outputs=sorted(os.listdir(OUT)), figures=sorted(os.listdir(FIG)),
                resume='Set PACA_ROOT to a writable folder and run all cells; set PACA_FORCE=1 to recompute cached CSVs. '
                       'To use another dataset, build {subject: dict(X_raw=array, sess=array)} with chronological order and call run_grid().')
json.dump(manifest, open(os.path.join(OUT, 'reproducibility_manifest.json'), 'w'), indent=1, default=str)
print(json.dumps({k: manifest[k] for k in ['generated_utc', 'official_cmu_host', 'outputs']}, indent=1, default=str))

## 20. Revision experiments (response to the IEEE Access reviews of Access-2026-55189)

The sections above are the original, pre-specified study and are left unchanged so that every number in the first submission can be regenerated. This section adds the experiments requested by the two reviewers. **Everything here is post hoc relative to the original plan.** The revision analyses were written to `outputs/revision/revision_plan.json` (SHA-256 printed below) before any revision run was executed, but that file has no external timestamp either; we therefore report them as *revision analyses*, not as confirmatory tests.

| Reviewer point | What was asked | Where answered |
|---|---|---|
| R1-1, R2 (fixed τ_acc) | Compare at matched FAR; DET/ROC; how τ is selected and revalidated after each update | 20.2 |
| R1-2, R2 (power) | Qualify the null; equivalence test with a justified margin; more victims; power | 20.3 |
| R1-3, R2 (A1 makes labels-only true by construction) | Labels-only under non-zero attacker step-up pass and genuine failure; lockout; handover after a genuine step-up; selection effect | 20.4 |
| R1-7, R2 (attacker realism, stronger attacks) | Gate-aware, generative, physically realisable and stronger attackers | 20.5 |
| R2 (contrasts not single-factor) | Add policies so each contrast changes one component | 20.6 |
| R1-4 (privacy scope) | MIA on the adaptive model; what non-private outputs reveal; user's own records; composition across released models | 20.7 |
| R1-6, R2 (external validity) | Drift other than a practice effect; beyond the linear verifier; Touchalytics cannot separate defences; on-device cost | 20.8 |
| R1-5, R2 (reproducibility) | Code, configuration, seeds, partitions, attack implementation | 20.9 |
| All | Which claims survive | 20.10 |

### Revision library (tested module, written to disk; with all extensions switched off it reproduces `run_stream` bit-for-bit)

In [ ]:
%%writefile pacau_rev.py
# ==== pacau_rev: revision experiments (reviewer-driven extensions of pacau_core) ====
# Every extension is OFF by default: with default settings run_stream2() consumes exactly the same random
# numbers as pacau_core.run_stream() and reproduces its outputs bit-for-bit (asserted in the notebook).
import numpy as np, pandas as pd, time, os, multiprocessing as mp
from pacau_core import (Authenticator, Accountant, METHODS, CONDITIONS, ALPHA_GRID, DEFAULT_CFG, sigmoid, add_bias,
                        eer, auc, partition, build_run, attack_plan, sid, cmu_transform)

# ------------------------------------------------------------------ policies
METHODS2 = dict(METHODS)
# single-factor contrast completions (Reviewer 2): P vs B2p isolates the silent gate under DP with identical
# budget + pacing; P vs B3p isolates DP noise/clipping with an identical (accountant-driven) update schedule.
METHODS2['B2p_DP_selfupdate_paced'] = dict(METHODS['B2_DP_selfupdate'], pacing=True)
METHODS2['B3p_gate_sched'] = dict(METHODS['P_gate_DP_budget'], dp=False, sched=True)
# selection-effect control for labels-only updating: also challenge a random 5% of silently accepted samples
METHODS2['B1c_audit5'] = dict(METHODS['B1c_labels_only'], audit=0.05)
# grace-window labelling: after a passed step-up the next G presented samples are treated as verified
for k in ['B1c_labels_only', 'B2c_DP_labels_only', 'P_gate_DP_budget']:
    METHODS2[k + '_grace'] = dict(METHODS[k], grace=True)

CORE = ['B0_static', 'B1_selfupdate', 'B1b_double_thr', 'B1c_labels_only', 'B2_DP_selfupdate', 'B2p_DP_selfupdate_paced',
        'B2c_DP_labels_only', 'B3_gate_nonDP', 'B3p_gate_sched', 'P_gate_DP_budget']
KEY = ['B0_static', 'B1_selfupdate', 'B1c_labels_only', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']

# ------------------------------------------------------------------ attack conditions
CONDITIONS2 = dict(CONDITIONS)
CONDITIONS2.update({
    'C3n_oracle_motornoise_20': ('oracle_noisy', 0.20),   # C3 + human motor noise: physically realisable on a keyboard
    'C5_gateaware_20':          ('gate_aware', 0.20),     # white-box incl. novelty statistic and trust radius R
    'C6_generative_20':         ('generative', 0.20),     # samples from a generative model of the victim fitted on leaks
    'C3s_oracle_strong_40':     ('oracle', 0.40),         # stronger dose: 40% of stream, bursts twice as long (20 on CMU)
    'H_handover_sessionstart':  ('handover', 0.0),        # owner signs in (strong factor) then hands the device over: one natural burst per session
})
FAR_GRID = np.logspace(-3, 0, 25)          # DET grid (probe FAR) on which per-run FRR is stored

# ------------------------------------------------------------------ authenticator with operating-point revalidation
class Authenticator2(Authenticator):
    """Adds: (i) FAR-targeted threshold revalidation after every (re)fit on the public reference negatives,
    (ii) accountant-driven scheduling without noise (B3p), (iii) accountant composition across released models."""
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.tau_acc, self.tau_upd = self.c['tau_acc'], self.c['tau_upd']
    def fit(self, steps):
        super().fit(steps)
        if self.c.get('far_target'):
            neg = self.score(self.Xcal[self.ycal == 0])                 # public reference set only (bg_cal)
            ft = self.c['far_target']
            self.tau_acc = float(np.quantile(neg, 1 - ft))
            self.tau_upd = float(max(self.c['tau_upd'], np.quantile(neg, 1 - ft / 4)))
    def can_spend(self, t, horizon):
        if not ((self.m['dp'] or self.m.get('sched')) and self.m['budget']): return True
        if self.acct.eps(self.steps + self.c['upd_steps']) > self.c['eps_budget']: return False
        if self.m['pacing']:
            if not hasattr(self, 'min_gap'): self.min_gap = horizon / max(self.n_affordable(), 1)
            return (t - self.last_update_t) >= self.min_gap
        return True
    def admit(self, X, p, verified):
        c0 = self.c; self.c = dict(c0, tau_acc=self.tau_acc, tau_upd=self.tau_upd)
        try: return super().admit(X, p, verified)
        finally: self.c = c0

class ComposedAccountant(Accountant):
    """Privacy loss of one background participant who is a unit in K released models, each run for `steps` steps:
    RDP composes additively over the K*steps Poisson-subsampled Gaussian steps."""
    def __init__(self, q, sigma, delta, K):
        super().__init__(q, sigma, delta); self.K = K
    def eps(self, steps, method='rdp'):
        return super().eps(steps * self.K, method)

def sigma_for_population(K, target_eps_init, q=0.2, delta=1e-5, steps=150):
    """Smallest noise multiplier such that K models' enrolment training composes to <= target_eps_init."""
    lo, hi = 1.0, 200.0
    for _ in range(40):
        mid = (lo + hi) / 2
        if Accountant(q, mid, delta).eps(K * steps) > target_eps_init: lo = mid
        else: hi = mid
    return hi

# ------------------------------------------------------------------ motor noise (CMU feature layout H, DD, UD x10 + H.Return)
def motor_noise(x, rng, cv=0.08, floor=0.005):
    x = x.copy(); d = len(x)
    if d == 31:
        H = np.r_[np.arange(0, 30, 3), 30]; UD = np.arange(2, 30, 3); DD = np.arange(1, 30, 3)
        for idx in (H, UD): x[idx] = x[idx] + rng.normal(0, cv * np.abs(x[idx]) + floor)
        x[H] = np.maximum(x[H], 0.001)
        x[DD] = x[DD - 1] + x[DD + 1]                      # keep DD = H(prev key) + UD consistent
    else:
        x = x + rng.normal(0, cv * np.abs(x) + floor)
    return x

# ------------------------------------------------------------------ prequential simulation (extended)
def run_stream2(run, cfg, method, acct, cond, seed, collect_trace=False, scores=False):
    kind, rate = CONDITIONS2[cond]
    if cond == 'C3s_oracle_strong_40': cfg = dict(cfg, burst=2 * cfg['burst'])
    rng = np.random.default_rng(seed)
    out_rng = np.random.default_rng(seed + 17)
    aux = np.random.default_rng(seed * 7 + 12345)              # separate stream: audit, handover, noise, generative
    t0 = time.perf_counter()
    A = Authenticator2(run['anchor'], run['bg_units'], run['cal_g'], run['cal_bg'], cfg, method, acct, rng)
    A.fit(cfg['init_steps']); t_init = time.perf_counter() - t0
    psr_init = float(np.mean(A.score(run['att_hold']) >= A.tau_acc))
    Xs, ss = run['stream_X'], run['stream_s']; n = len(Xs); horizon = n
    plan = set(attack_plan(n, rate, cfg['burst'], seed + 7919 * sid(run['victim']))) if kind != 'handover' else set()
    apool = run['att_pool_raw']; aptr = 0; alpha = 0.0; vleak = run['leak_raw']
    if kind == 'generative':                                    # diagonal Gaussian victim model in transformed space
        Lt = run['transform'](vleak); g_mu, g_sd = Lt.mean(0), Lt.std(0) + 0.05 * np.abs(Lt.mean(0)) + 1e-3
        def inv_t(Y): return np.sign(Y) * 0.01 * np.expm1(np.abs(Y)) if run['transform'] is cmu_transform else np.sign(Y) * np.expm1(np.abs(Y))
    mmin = cfg['m_min'] if method['silent'] == 'risk' or method['pacing'] else 1
    G_win = cfg.get('grace_window', 10) if method.get('grace') else 0
    rec = []; ep = []; upd_t = []; ckpt = []; inf_t = []; tcount = [0]; grace_left = [0]; pending_handover = [False]
    def flush():
        if method['adaptive'] and ep:
            X = np.vstack([e[0] for e in ep]); p = np.array([e[1] for e in ep])
            ver = np.array([e[2] for e in ep]); f = np.array([e[3] for e in ep]); idx = [e[4] for e in ep]
            ok = A.admit(X, p, ver)
            if ok.any():
                A.queue(X[ok], f[ok])
                for i in np.where(ok)[0]: rec[idx[i]]['admitted'] = True
            t = time.perf_counter()
            if A.maybe_update(tcount[0], horizon, mmin): upd_t.append(time.perf_counter() - t)
        ep.clear()
    def present(x, is_att, sess):
        t = time.perf_counter(); p = float(A.score(x[None])[0]); inf_t.append(time.perf_counter() - t)
        in_grace = G_win and grace_left[0] > 0
        audit = (not in_grace) and method.get('audit', 0.0) > 0 and p >= A.tau_acc and aux.random() < method['audit']
        if in_grace: dec, ver = 'accept', False                 # trust window: no re-challenge, label inherited
        elif p >= A.tau_acc and not audit: dec, ver = 'accept', False
        else:
            passed = out_rng.random() < (cfg['p_att_pass'] if is_att else cfg['p_gen_pass'])
            dec, ver = ('stepup_pass' if passed else 'stepup_fail'), passed
            if audit: dec = 'audit_' + dec
        lab = ver
        if G_win:
            if in_grace: lab = True; grace_left[0] -= 1
            elif ver: grace_left[0] = G_win
        rec.append(dict(p=p, dec=dec, att=is_att, sess=int(sess), admitted=False, verified=ver, labelled=lab,
                        nov=float(A.novelty(x[None])[0])))
        if lab: A.add_verified(x[None])
        ep.append((x[None], p, lab, is_att, len(rec) - 1)); tcount[0] += 1
        if len(ep) >= cfg['W']: flush()
        return dec
    def burst(sess):
        nonlocal aptr, alpha
        outs = []
        for k in range(cfg['burst']):
            if kind in ('zero', 'handover'): xa = apool[aptr % len(apool)]
            elif kind in ('oracle', 'oracle_noisy', 'gate_aware', 'generative'):
                base = vleak[k % len(vleak)]
                if kind == 'generative': base = inv_t(aux.normal(g_mu, g_sd))
                cand = (1 - ALPHA_GRID[:, None]) * base + ALPHA_GRID[:, None] * apool[aptr % len(apool)]
                Zc = run['Z'](run['transform'](cand)); pc = A.score(Zc)
                okc = pc >= A.tau_upd
                if kind == 'gate_aware' and np.isfinite(A.R): okc &= A.novelty(Zc) <= A.R
                okc = np.where(okc)[0]; xa = cand[okc.max()] if len(okc) else cand[0]
                if kind == 'oracle_noisy': xa = motor_noise(xa, aux)
            else: xa = (1 - alpha) * vleak[k % len(vleak)] + alpha * apool[aptr % len(apool)]
            aptr += 1
            o = present(run['Z'](run['transform'](xa[None]))[0], True, sess); outs.append(o)
            if o.endswith('stepup_fail'): break
        if kind == 'adaptive':
            acc = np.mean([o == 'accept' for o in outs]) if len(outs) == cfg['burst'] else 0.0
            alpha = min(1.0, alpha + cfg['adapt_step_up']) if acc >= 0.8 else max(0.0, alpha - cfg['adapt_step_down'])
    prev_s = None; ho_done = 0; probe = run['probe']
    for i in range(n):
        if ss[i] != prev_s:
            g = A.score(Xs[ss == ss[i]]); im = A.score(probe)
            row = dict(sess=int(ss[i]), eer=eer(g, im), frr=float(np.mean(g < A.tau_acc)),
                       far=float(np.mean(im >= A.tau_acc)), eps=A.eps() if method['dp'] else np.nan,
                       n_updates=A.n_updates, tau_acc=A.tau_acc)
            ims = np.sort(im)[::-1]
            for ft in (0.01, 0.05, 0.10):                      # oracle matched-FAR threshold on probes
                k_ = max(int(np.floor(ft * len(ims))), 1); thr = ims[k_ - 1] + 1e-12
                row[f'frr_at_far{int(ft*100):02d}'] = float(np.mean(g < thr))
            row['det'] = np.array([np.mean(g < (ims[max(int(np.floor(f * len(ims))), 1) - 1] + 1e-12)) for f in FAR_GRID])
            ckpt.append(row); prev_s = ss[i]
            if G_win: grace_left[0] = G_win                  # session sign-in with the strong factor opens a grace window
            if kind == 'handover': burst(ss[i])               # owner signs in, then hands the device over
        if i in plan: burst(ss[i])
        present(Xs[i], False, ss[i])
    flush()
    R = pd.DataFrame(rec); G = R[~R.att]; Aa = R[R.att]
    late = G.sess >= (G.sess.max() - 2)
    delays = []
    if len(Aa):
        idx = np.where(R.att.values)[0]; starts = [idx[0]] + [j for a_, j in zip(idx[:-1], idx[1:]) if j != a_ + 1]
        for st in starts:
            d = 0
            while st + d < len(R) and R.att.values[st + d] and R.dec.values[st + d] == 'accept': d += 1
            delays.append(d)
    ck = pd.DataFrame(ckpt); det_late = np.vstack(ck.det.iloc[-3:].values).mean(0); ck = ck.drop(columns='det')
    Gad = G[G.admitted]
    out = dict(
        online_FRR=float(np.mean(G.dec != 'accept')), online_FRR_late=float(np.mean(G[late].dec != 'accept')),
        genuine_stepup_rate=float(np.mean(G.dec != 'accept')),
        genuine_lockout_rate=float(np.mean(G.dec.str.endswith('stepup_fail'))),
        attack_accept_rate=float(np.mean(Aa.dec == 'accept')) if len(Aa) else np.nan,
        attack_samples_accepted=int((Aa.dec == 'accept').sum()) if len(Aa) else 0,
        attack_stepup_passed=int(Aa.dec.str.endswith('stepup_pass').sum()) if len(Aa) else 0,
        detection_delay=float(np.mean(delays)) if delays else np.nan,
        n_updates=A.n_updates, n_admitted=int(R.admitted.sum()),
        n_admitted_attack=int(Aa.admitted.sum()) if len(Aa) else 0,
        n_admitted_verified=int(G[G.admitted].verified.sum()) if len(G) else 0,
        buffer_contam_final=float(A.bufflag.mean()) if len(A.bufflag) else 0.0,
        PSR_init=psr_init, PSR_final=float(np.mean(A.score(run['att_hold']) >= A.tau_acc)),
        eps_final=A.eps() if method['dp'] else np.nan,
        eps_final_pld=acct.eps(A.steps, 'pld') if method['dp'] else np.nan,
        max_unit_grad_norm=float(np.max(A.clip_stats)) if A.clip_stats else np.nan,
        ckpt_EER_mean=float(ck.eer.iloc[1:].mean()), ckpt_EER_late=float(ck.eer.iloc[-3:].mean()),
        ckpt_FAR_late=float(ck.far.iloc[-3:].mean()), ckpt_FRR_late=float(ck.frr.iloc[-3:].mean()),
        FRR_at_FAR01_late=float(ck.frr_at_far01.iloc[-3:].mean()), FRR_at_FAR05_late=float(ck.frr_at_far05.iloc[-3:].mean()),
        FRR_at_FAR10_late=float(ck.frr_at_far10.iloc[-3:].mean()), tau_acc_final=A.tau_acc, tau_upd_final=A.tau_upd,
        p_admitted_genuine=float(Gad.p.mean()) if len(Gad) else np.nan, nov_admitted_genuine=float(Gad.nov.mean()) if len(Gad) else np.nan,
        p_genuine_all=float(G.p.mean()), nov_genuine_all=float(G.nov.mean()),
        t_init_s=t_init, t_update_s=float(np.mean(upd_t)) if upd_t else np.nan,
        t_infer_us=float(np.median(inf_t) * 1e6), steps=A.steps, n_stream=len(R), n_attack=int(len(Aa)),
        upd_per_session=','.join(str(int(v)) for v in np.diff(np.r_[ck.n_updates.values, A.n_updates])),
    )
    for j, v in enumerate(det_late): out[f'det{j:02d}'] = float(v)
    if collect_trace: return out, R, ck, A
    return out, ck

# ------------------------------------------------------------------ data variants
def rff_wrap(run, D=256, ls=0.5, seed=2024):
    """Non-linear verifier: fixed, data-independent random Fourier features approximating an RBF kernel with
    lengthscale ls*sqrt(d) on the standardised features. D and ls were chosen on the pilot partition (seed 999) by
    static-model EER only (D=256, ls=0.5: 0.135 vs 0.145 for the linear verifier). Being data-independent, the map
    leaves the DP guarantee unchanged; the model then has D+1 weights."""
    d = run['anchor'].shape[1]; r = np.random.default_rng(seed)
    W = r.normal(0, 1 / (ls * np.sqrt(d)), size=(d, D)); b = r.uniform(0, 2 * np.pi, D)
    phi = lambda Z: np.sqrt(2.0 / D) * np.cos(Z @ W + b)
    out = dict(run); Z0 = run['Z']
    out['Z'] = lambda X: phi(Z0(X))
    for k in ('anchor', 'cal_g', 'stream_X', 'cal_bg', 'att_hold', 'probe'): out[k] = phi(run[k])
    out['bg_units'] = [phi(U) for U in run['bg_units']]
    return out

def inv_cmu(Y): return np.sign(Y) * 0.01 * np.expm1(np.abs(Y))

def drift_variant(data, mode, seed=0, transform=cmu_transform, inv=inv_cmu):
    """mode='detrended': remove each subject's per-session mean shift (practice effect) in transformed space.
    mode='randomwalk': detrended + a per-subject, per-feature Gaussian random walk across sessions whose
    session-8 displacement is matched to the original data's median displacement (non-monotone 'ageing')."""
    r = np.random.default_rng(seed); out = {}
    disp_orig = []
    for s, v in data.items():
        Y = transform(v['X_raw']); ss = v['sess']; m1, sd1 = Y[ss == 1].mean(0), Y[ss == 1].std(0) + 1e-6
        disp_orig.append(np.sqrt(np.mean(((Y[ss == ss.max()].mean(0) - m1) / sd1) ** 2)))
    target = float(np.median(disp_orig))
    for s, v in data.items():
        Y = transform(v['X_raw']).copy(); ss = v['sess']; m1 = Y[ss == 1].mean(0); sd1 = Y[ss == 1].std(0) + 1e-6
        for k in np.unique(ss): Y[ss == k] += m1 - Y[ss == k].mean(0)
        if mode == 'randomwalk':
            K = ss.max(); steps = r.normal(0, 1, size=(K - 1, Y.shape[1]))
            walk = np.vstack([np.zeros(Y.shape[1]), np.cumsum(steps, 0)])
            scale = target / (np.sqrt(np.mean(walk[-1] ** 2)) + 1e-9)
            for k in range(2, K + 1): Y[ss == k] += walk[k - 1] * scale * sd1
        out[s] = dict(X_raw=inv(Y), sess=ss)
    return out, target

# ------------------------------------------------------------------ parallel grid
_G = {}
def _job(args):
    sd, v, att, part = args['sd'], args['v'], args['att'], args['part']
    data, transform, cfg, methods, conds, tag = _G['data'], _G['transform'], args['cfg'], args['methods'], args['conds'], args['tag']
    key = (id(_G['acct_factory']), cfg['q'], cfg['sigma'], cfg['delta'])
    cache = _G.setdefault('acct_cache', {})
    if key not in cache: cache[key] = _G['acct_factory'](cfg)        # per-process accountant cache (RDP/PLD are costly)
    acct = cache[key]
    run = build_run(data, transform, part, v, att, cfg, seed=sd)
    if _G.get('wrap'): run = _G['wrap'](run)
    rows, cks = [], []
    for cond in conds:
        for mn in methods:
            out, ck = run_stream2(run, cfg, METHODS2[mn], acct, cond, seed=sd)
            rows.append(dict(tag=tag, seed=sd, victim=str(v), attacker=str(att), cond=cond, method=mn, **out))
            cks.append(ck.assign(tag=tag, seed=sd, victim=str(v), cond=cond, method=mn))
    return rows, cks

def default_acct(cfg): return Accountant(cfg['q'], cfg['sigma'], cfg['delta'])

def run_grid2(data, transform, seeds, methods, conds, cfg, part_sizes, tag, victims_fixed=None, n_jobs=2,
              acct_factory=default_acct, wrap=None, max_victims=None):
    jobs = []
    for sd in seeds:
        if victims_fixed is None: part = partition(list(data), *part_sizes, seed=sd)
        else:
            others = [s for s in data if s not in victims_fixed]; part = partition(others, *part_sizes, seed=sd); part['victims'] = list(victims_fixed)
        for v in part['victims'][:max_victims]:
            jobs.append(dict(sd=sd, v=v, att=part['attackers'][sid(v) % len(part['attackers'])], part=part, cfg=cfg,
                             methods=methods, conds=conds, tag=tag))
    _G.update(data=data, transform=transform, acct_factory=acct_factory, wrap=wrap, acct_cache={})
    t0 = time.time()
    if n_jobs > 1:
        with mp.get_context('fork').Pool(n_jobs) as pool: res = pool.map(_job, jobs, chunksize=1)
    else: res = [_job(j) for j in jobs]
    rows = [r for rr, _ in res for r in rr]; cks = [c for _, cc in res for c in cc]
    print(f'[{tag}] {len(rows)} runs in {time.time() - t0:.0f}s', flush=True)
    return pd.DataFrame(rows), pd.concat(cks, ignore_index=True)

# ------------------------------------------------------------------ statistics
def tost_paired(d, margin, alpha=0.05):
    """Two one-sided Wilcoxon signed-rank tests of |mean paired difference| < margin (equivalence)."""
    from scipy.stats import wilcoxon
    d = np.asarray(d, float); d = d[~np.isnan(d)]
    p_lo = wilcoxon(d + margin, alternative='greater').pvalue      # H0: d <= -margin
    p_hi = wilcoxon(d - margin, alternative='less').pvalue         # H0: d >= +margin
    from scipy.stats import ttest_1samp
    t_lo = ttest_1samp(d, -margin, alternative='greater').pvalue; t_hi = ttest_1samp(d, margin, alternative='less').pvalue
    return dict(margin=margin, p_tost_wilcoxon=max(p_lo, p_hi), p_tost_t=max(t_lo, t_hi),
                ci90_lo=float(np.mean(d) - 1.645 * d.std(ddof=1) / np.sqrt(len(d))),
                ci90_hi=float(np.mean(d) + 1.645 * d.std(ddof=1) / np.sqrt(len(d))),
                equivalent=max(p_lo, p_hi) < alpha)

def mde(sd, n, alpha=0.05, power=0.8, are=3 / np.pi):
    from scipy.stats import norm
    return float((norm.ppf(1 - alpha / 2) + norm.ppf(power)) * sd / np.sqrt(n * are))

In [ ]:
import importlib, pacau_rev; importlib.reload(pacau_rev); from pacau_rev import *
N_JOBS = int(os.environ.get('PACA_JOBS', min(os.cpu_count() or 1, 8)))
SEEDS_REP = [5, 6, 7, 8, 9] if not SMOKE else [5]
REV = os.path.join(OUT, 'revision'); FIGR = os.path.join(FIG, 'revision')
for d_ in (REV, FIGR): os.makedirs(d_, exist_ok=True)
MARGIN = 0.05
P_ATT = [0.0, 0.01, 0.05, 0.10, 0.20, 1.0]; P_GEN = [1.0, 0.95, 0.90]; FAR_TARGETS = [0.02, 0.05, 0.10]
STEP_POL = ['B1_selfupdate', 'B1c_labels_only', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']
HAND_POL = ['B1_selfupdate', 'B1c_labels_only', 'B1c_labels_only_grace', 'B2c_DP_labels_only', 'B2c_DP_labels_only_grace',
            'P_gate_DP_budget', 'P_gate_DP_budget_grace']
NEW_ATTACKS = ['C3n_oracle_motornoise_20', 'C5_gateaware_20', 'C6_generative_20']
revision_plan = dict(
    status='post hoc relative to the original plan; written before any revision run; no external timestamp',
    margin=MARGIN,
    margin_rationale='5 pp is the pre-registered practical margin; it is also of the order of the static verifier\'s late '
                     'impostor acceptance (0.068), so a smaller change in attacker acceptance would not alter a deployment decision.',
    analyses={
      'R-OP': 'Operating point: CORE policies at fixed tau_acc (FRR at matched probe FAR 1/5/10%, per-run DET), and KEY policies with '
              'tau_acc re-selected after every (re)fit so that FAR on the public reference set equals 2/5/10%. Endpoints: online '
              'challenge rate (C0), realised late probe FAR, attack gain (C3). Tests: P-B0, B1c-B0, P-B1c challenge rate, Holm within target.',
      'R-PRIM': 'Primary contrast: TOST (+-margin) on gain(P)-gain(B1) under C3; replication on 5 new partitions (seeds 5-9) and pooled '
                '10 partitions; stronger attack C3s (40%, bursts x2) where B1 gain exceeds the MDE.',
      'R-STEP': 'Step-up realism: attack gain, attacker admissions and genuine lock-outs for STEP_POL across p_att in P_ATT (p_gen=1) and '
                'p_gen in P_GEN (p_att in {0,0.05}), under C2 (natural, label-seeking) and C3 (crafted). Handover after sign-in with '
                'per-sample vs trust-window (G=10) labels. Selection-effect control B1c_audit5.',
      'R-ATT': 'Attack suite: CORE policies under C3n (motor noise), C5 (gate-aware), C6 (generative), C3s (strong).',
      'R-SF': 'Single-factor contrasts: B1c-B1 (silent admission), B3-B1 (gate, no DP), P-B2p (gate | DP, same budget+pacing), '
              'P-B3p (DP | gate, same schedule), B2-B1 (DP+budget); attack gain under C3, C3s, C5; Holm within attack.',
      'R-PRIV': 'Participant- and record-level MIA on final adaptive models (C0); template recovery from leaked novelty statistics; '
                'update-log side channel AUC; composition across K=20 released models and utility at the population-calibrated sigma.',
      'R-GEN': 'Random-Fourier-feature verifier (D=256, pilot-tuned); drift regimes (practice effect removed; matched-magnitude random walk); '
               'Touchalytics with the strong attack; analytic on-device cost.'},
    seeds=dict(main=SEEDS_MAIN, replication=SEEDS_REP), core=CORE, key=KEY, step_pol=STEP_POL, hand_pol=HAND_POL,
    p_att=P_ATT, p_gen=P_GEN, far_targets=FAR_TARGETS, new_attacks=NEW_ATTACKS)
PLAN = os.path.join(REV, 'revision_plan.json')
if os.path.exists(PLAN):
    old = json.load(open(PLAN)); same = {k: old[k] for k in old if k != 'written_utc'} == json.loads(json.dumps(revision_plan, default=str))
    print('existing revision plan kept, written', old['written_utc'], '| identical to current code:', same); assert same
else:
    json.dump(dict(written_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), **revision_plan), open(PLAN, 'w'), indent=1, default=str)
print('revision plan SHA-256:', hashlib.sha256(open(PLAN, 'rb').read()).hexdigest())
# the extended simulator reproduces the original one bit-for-bit when no extension is switched on
for sd in [0, 3]:
    part = partition(list(cmu), 14, 6, 11, seed=sd)
    for v in part['victims'][:2]:
        run = build_run(cmu, cmu_transform, part, v, part['attackers'][sid(v) % 11], DEFAULT_CFG, seed=sd)
        for mn in ['B1_selfupdate', 'B1c_labels_only', 'P_gate_DP_budget']:
            for c in ['C0_drift_only', 'C3_confaware_oracle_20', 'C4_greybox_feedback_20']:
                a_, _ = run_stream(run, DEFAULT_CFG, METHODS[mn], acct, c, seed=sd); b_, _ = run_stream2(run, DEFAULT_CFG, METHODS2[mn], acct, c, seed=sd)
                assert all((a_[k] == b_[k]) or (np.isnan(a_[k]) and np.isnan(b_[k])) for k in a_ if not k.startswith('t_')), (sd, v, mn, c)
print('run_stream2 == run_stream (bit-identical) on 36 runs;', 'N_JOBS =', N_JOBS)

def cached(name, fn):
    p = os.path.join(REV, name)
    if os.path.exists(p) and not FORCE_RERUN: return pd.read_csv(p, dtype={'victim': str})
    df = fn(); df.to_csv(p, index=False); return df
def S2(df, metric, method, cond, **flt):
    g = df[(df.method == method) & (df.cond == cond)]
    for k, v in flt.items(): g = g[np.isclose(g[k], v)] if isinstance(v, float) else g[g[k] == v]
    return g.groupby('victim')[metric].mean()
def G2(df, method, attack, base='C0_drift_only', base_df=None, **flt):
    return S2(df, 'PSR_final', method, attack, **flt) - S2(base_df if base_df is not None else df, 'PSR_final', method, base, **flt)
def practical(lo, hi, m=MARGIN):
    if lo > 0 or hi < 0:
        if min(abs(lo), abs(hi)) >= m: return 'practically significant (CI beyond margin)'
        if max(abs(lo), abs(hi)) < m: return 'significant but negligible (CI within margin)'
        return 'significant; size uncertain vs margin'
    return 'no meaningful difference (CI within margin)' if (lo > -m and hi < m) else 'inconclusive (CI includes 0 and margin)'
def contrast_table(pairs, df_fn, label_fmt, family):
    rows = [paired_test(a, b, lab) for lab, a, b in pairs]
    t = pd.DataFrame(rows); t['p_holm'] = holm(t.p_value.values); t['family'] = family
    t['interpretation'] = [practical(r.ci_lo, r.ci_hi) for r in t.itertuples()]
    return t

### 20.1 Revision grids
All grids use the same partitions, victims, attackers, seeds and default settings as the main study (seeds 0–4), except the replication grid (seeds 5–9, new partitions). Results are cached in `outputs/revision/`; set `PACA_FORCE=1` to recompute.

In [ ]:
COND3 = ['C0_drift_only', 'C3_confaware_oracle_20', 'C3s_oracle_strong_40']
core = cached('core_fixed.csv', lambda: run_grid2(cmu, cmu_transform, SEEDS_MAIN, CORE, COND3, DEFAULT_CFG, (14, 6, 11), 'core', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)[0])
rep = cached('replication_seeds5to9.csv', lambda: run_grid2(cmu, cmu_transform, SEEDS_REP, CORE, COND3, DEFAULT_CFG, (14, 6, 11), 'rep', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)[0])
att = cached('attack_suite.csv', lambda: run_grid2(cmu, cmu_transform, SEEDS_MAIN, CORE, NEW_ATTACKS, DEFAULT_CFG, (14, 6, 11), 'attacks', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)[0])
def _far():
    parts = []
    for ft in FAR_TARGETS:
        r_, _ = run_grid2(cmu, cmu_transform, SEEDS_MAIN, KEY, ['C0_drift_only', 'C3_confaware_oracle_20'], dict(DEFAULT_CFG, far_target=ft), (14, 6, 11), f'far{ft}', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)
        parts.append(r_.assign(far_target=ft))
    return pd.concat(parts, ignore_index=True)
far = cached('far_revalidated.csv', _far)
def _step():
    parts = []
    for pa in P_ATT:
        r_, _ = run_grid2(cmu, cmu_transform, SEEDS_MAIN, STEP_POL, ['C2_zero_effort_30', 'C3_confaware_oracle_20'], dict(DEFAULT_CFG, p_att_pass=pa), (14, 6, 11), f'patt{pa}', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)
        parts.append(r_.assign(p_att=pa, p_gen=1.0))
    for pg in P_GEN[1:]:
        r0, _ = run_grid2(cmu, cmu_transform, SEEDS_MAIN, STEP_POL, ['C0_drift_only'], dict(DEFAULT_CFG, p_gen_pass=pg), (14, 6, 11), f'pgen{pg}', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)
        parts.append(r0.assign(p_att=np.nan, p_gen=pg))
        for pa in [0.0, 0.05]:
            r_, _ = run_grid2(cmu, cmu_transform, SEEDS_MAIN, STEP_POL, ['C2_zero_effort_30', 'C3_confaware_oracle_20'], dict(DEFAULT_CFG, p_gen_pass=pg, p_att_pass=pa), (14, 6, 11), f'pgen{pg}_patt{pa}', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)
            parts.append(r_.assign(p_att=pa, p_gen=pg))
    return pd.concat(parts, ignore_index=True)
step = cached('stepup_realism.csv', _step)
hand = cached('handover.csv', lambda: run_grid2(cmu, cmu_transform, SEEDS_MAIN, HAND_POL + ['B1c_audit5'], ['C0_drift_only', 'H_handover_sessionstart', 'C3_confaware_oracle_20'], DEFAULT_CFG, (14, 6, 11), 'handover', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)[0])
print({k: len(v) for k, v in dict(core=core, rep=rep, attacks=att, far=far, step=step, handover=hand).items()})

In [ ]:
# reproduction check: overlapping cells of the revision grid equal the original main grid exactly
chk = core.merge(res, on=['seed', 'victim', 'cond', 'method'], suffixes=('', '_orig'))
for m_ in ['PSR_final', 'online_FRR', 'ckpt_EER_late', 'ckpt_FAR_late', 'n_admitted_attack']:
    assert np.allclose(chk[m_], chk[m_ + '_orig'], equal_nan=True), m_
print(f'revision grid reproduces {len(chk)} overlapping runs of the original grid exactly')

### 20.2 Operating point: friction at matched false acceptance (R1-1, R2)
Two complementary analyses. (a) *Oracle matched FAR*: at every pre-session checkpoint, the threshold is set so that exactly 1/5/10% of the disjoint probe impostors are accepted, and the genuine rejection of the coming session is recorded (also on a 25-point DET grid). This is the textbook comparison of verifiers at equal security, but it uses evaluation impostors and cannot be deployed. (b) *Deployable revalidation*: after the initial fit and after **every** update, τ_acc is re-selected on the public reference set (bg_cal, disjoint from training, attackers and probes) as the score quantile that yields a target reference FAR (2/5/10%); the silent-admission threshold becomes max(0.8, quantile at FAR/4). We then measure the online challenge rate, the *realised* probe FAR and the attack gain.

In [ ]:
# (a) oracle matched FAR from the fixed-threshold grid
op_rows = []
for m_ in CORE:
    r_ = dict(method=m_)
    for col in ['online_FRR', 'ckpt_FAR_late', 'ckpt_EER_late', 'FRR_at_FAR01_late', 'FRR_at_FAR05_late', 'FRR_at_FAR10_late']:
        s_ = S2(core, col, m_, 'C0_drift_only'); lo_, hi_ = boot_ci_mean(s_.values); r_[col] = s_.mean(); r_[col + '_ci'] = f'[{lo_:.3f}, {hi_:.3f}]'
    op_rows.append(r_)
op = pd.DataFrame(op_rows); op.to_csv(os.path.join(REV, 'operating_point_oracle.csv'), index=False)
display(op[['method', 'online_FRR', 'ckpt_FAR_late', 'ckpt_EER_late', 'FRR_at_FAR01_late', 'FRR_at_FAR05_late', 'FRR_at_FAR10_late']].round(3))
op_tests = contrast_table([(f'FRR@FAR{t_}% late: {a_} - {b_}', S2(core, f'FRR_at_FAR{t_}_late', a_, 'C0_drift_only'), S2(core, f'FRR_at_FAR{t_}_late', b_, 'C0_drift_only'))
                           for t_ in ['01', '05', '10'] for a_, b_ in [('P_gate_DP_budget', 'B0_static'), ('B1c_labels_only', 'B0_static'), ('P_gate_DP_budget', 'B1c_labels_only')]],
                          None, None, 'oracle matched FAR (Holm over 9)')
display(op_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
# (b) deployable revalidation
far_rows = []
for ft in FAR_TARGETS:
    f_ = far[np.isclose(far.far_target, ft)]
    for m_ in KEY:
        ch = S2(f_, 'online_FRR', m_, 'C0_drift_only'); fl = S2(f_, 'ckpt_FAR_late', m_, 'C0_drift_only'); g_ = G2(f_, m_, 'C3_confaware_oracle_20')
        far_rows.append(dict(far_target=ft, method=m_, challenge=ch.mean(), challenge_ci=boot_ci_mean(ch.values), late_probe_FAR=fl.mean(),
                             late_probe_FAR_ci=boot_ci_mean(fl.values), gain_C3=g_.mean(), gain_C3_ci=boot_ci_mean(g_.values),
                             tau_final=S2(f_, 'tau_acc_final', m_, 'C0_drift_only').mean(), eps=S2(f_, 'eps_final', m_, 'C0_drift_only').mean()))
fr = pd.DataFrame(far_rows); fr.to_csv(os.path.join(REV, 'operating_point_revalidated.csv'), index=False)
display(fr.drop(columns=[c for c in fr if c.endswith('_ci')]).round(3))
far_tests = []
for ft in FAR_TARGETS:
    f_ = far[np.isclose(far.far_target, ft)]
    t_ = contrast_table([(f'FAR{ft:.2f} challenge: {a_} - {b_}', S2(f_, 'online_FRR', a_, 'C0_drift_only'), S2(f_, 'online_FRR', b_, 'C0_drift_only'))
                         for a_, b_ in [('P_gate_DP_budget', 'B0_static'), ('B1c_labels_only', 'B0_static'), ('P_gate_DP_budget', 'B1c_labels_only'), ('B1_selfupdate', 'B0_static')]]
                        + [(f'FAR{ft:.2f} realised probe FAR: P - B0', S2(f_, 'ckpt_FAR_late', 'P_gate_DP_budget', 'C0_drift_only'), S2(f_, 'ckpt_FAR_late', 'B0_static', 'C0_drift_only'))],
                        None, None, f'revalidated FAR {ft}')
    far_tests.append(t_)
far_tests = pd.concat(far_tests, ignore_index=True); far_tests.to_csv(os.path.join(REV, 'tests_operating_point.csv'), index=False)
display(far_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
# figure: mean late DET (C0) and revalidated friction vs realised FAR
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.9))
for m_, ls in [('B0_static', '-'), ('B1_selfupdate', '-'), ('B1c_labels_only', '-.'), ('B2c_DP_labels_only', '--'), ('B3_gate_nonDP', '-'), ('P_gate_DP_budget', '-')]:
    g_ = core[(core.method == m_) & (core.cond == 'C0_drift_only')].groupby('victim')[[f'det{j:02d}' for j in range(len(FAR_GRID))]].mean().mean()
    ax[0].plot(FAR_GRID, g_.values, ls, label=m_)
ax[0].set(xscale='log', yscale='log', xlim=(5e-3, 1), ylim=(5e-3, 1), xlabel='Probe impostor acceptance (FAR)', ylabel='Genuine rejection (FRR)', title='(a) Late-session DET, drift only')
ax[0].legend(fontsize=5.5)
for m_ in KEY:
    q_ = fr[fr.method == m_]; ax[1].plot(q_.late_probe_FAR, q_.challenge, 'o-', ms=3, label=m_)
    a_ = op[op.method == m_]
    ax[1].scatter(a_.ckpt_FAR_late, a_.online_FRR, marker='x', s=18, color=ax[1].lines[-1].get_color())
ax[1].set(xlabel='Realised late probe FAR', ylabel='Online genuine challenge rate', title='(b) Revalidated τ (o) vs fixed τ=0.5 (x)'); ax[1].legend(fontsize=5.5)
plt.tight_layout(); plt.savefig(os.path.join(FIGR, 'figR1_operating_point.png')); plt.show()

### 20.3 The primary contrast: equivalence, power, replication and a stronger attack (R1-2, R2)

In [ ]:
prim_d = (gain('P_gate_DP_budget') - gain('B1_selfupdate')).dropna()
tost0 = tost_paired(prim_d.values, MARGIN)
sd_d = prim_d.std(ddof=1)
power_tab = pd.DataFrame([dict(n=n_, sd=sd_d, MDE_80=mde(sd_d, n_), MDE_90=mde(sd_d, n_, power=0.9)) for n_ in [47, 51, 100, 200]])
print(f'Original primary: mean {prim_d.mean():.4f}, 90% CI [{tost0["ci90_lo"]:.3f}, {tost0["ci90_hi"]:.3f}], TOST p (Wilcoxon) = {tost0["p_tost_wilcoxon"]:.3f}, '
      f'TOST p (t) = {tost0["p_tost_t"]:.3f} -> equivalence within +-{MARGIN} {"shown" if tost0["equivalent"] else "NOT shown"}')
print(f"B1 gain under C3 = {gain('B1_selfupdate').mean():.3f} vs MDE(n=47) = {mde(sd_d, 47):.3f}: even a perfect defence (gain 0) would be "
      f"{'below' if gain('B1_selfupdate').mean() < mde(sd_d, 47) else 'above'} the MDE"); display(power_tab.round(3))
# replication (new partitions) and pooled 10-partition analysis, C3 and the strong attack C3s
allp = pd.concat([core, rep], ignore_index=True)
prim_rows = []
for lab, df_ in [('seeds 0-4 (original partitions)', core), ('seeds 5-9 (new partitions)', rep), ('pooled 10 partitions', allp)]:
    for atk in ['C3_confaware_oracle_20', 'C3s_oracle_strong_40']:
        dP, dB = G2(df_, 'P_gate_DP_budget', atk), G2(df_, 'B1_selfupdate', atk); d_ = (dP - dB).dropna()
        pt = paired_test(dP, dB, f'{lab} | {atk}: gain(P) - gain(B1)'); tt = tost_paired(d_.values, MARGIN)
        prim_rows.append(dict(**pt, gain_B1=dB.mean(), gain_P=dP.mean(), gain_B1c=G2(df_, 'B1c_labels_only', atk).mean(),
                              gain_B3=G2(df_, 'B3_gate_nonDP', atk).mean(), MDE=mde(d_.std(ddof=1), len(d_)), p_TOST=tt['p_tost_wilcoxon'],
                              interpretation=practical(pt['ci_lo'], pt['ci_hi'])))
prim = pd.DataFrame(prim_rows); prim.to_csv(os.path.join(REV, 'primary_replication_power.csv'), index=False)
display(prim[['comparison', 'n_subjects', 'gain_B1', 'gain_P', 'gain_B3', 'gain_B1c', 'mean_diff', 'ci_lo', 'ci_hi', 'p_value', 'MDE', 'p_TOST', 'interpretation']].round(4))

### 20.4 Step-up realism: attackers who sometimes pass, users who sometimes fail, and handover (R1-3, R2)
A rational attacker against labels-only updating does *not* craft confident samples (those are accepted silently and never labelled); it presents its own natural behaviour **in order to be challenged**, and every passed challenge writes a labelled attacker sample into the template. We therefore evaluate both the natural, label-seeking attacker (C2, 30%) and the crafted one (C3), over attacker pass rates p_att ∈ {0, 1, 5, 10, 20, 100}% and genuine pass rates p_gen ∈ {100, 95, 90}%. A failed genuine step-up locks the session (counted as a lock-out) and yields no label. *Handover*: the owner signs in with the strong factor at the start of each session and hands the device to the attacker, who types naturally for one burst. With **per-sample** labels only samples that themselves passed a challenge are labelled; with a **trust window** (common in practice), a passed strong authentication suppresses re-challenges and labels the next G = 10 samples. *Selection effect*: B1c learns only from challenged, i.e. low-confidence, genuine samples; `B1c_audit5` additionally challenges a random 5% of silently accepted samples, giving unbiased labels.

In [ ]:
def gain_step(df, m_, atk, pa, pg):
    base = S2(df, 'PSR_final', m_, 'C0_drift_only', p_gen=pg) if pg < 1 else S2(core, 'PSR_final', m_, 'C0_drift_only')
    return S2(df, 'PSR_final', m_, atk, p_att=pa, p_gen=pg) - base
srows = []
for atk in ['C2_zero_effort_30', 'C3_confaware_oracle_20']:
    for pg in P_GEN:
        for pa in (P_ATT if pg == 1.0 else [0.0, 0.05]):
            for m_ in STEP_POL:
                g_ = gain_step(step, m_, atk, pa, pg); lo_, hi_ = boot_ci_mean(g_.values)
                srows.append(dict(attack=atk, p_gen=pg, p_att=pa, method=m_, gain=g_.mean(), lo=lo_, hi=hi_,
                                  att_admitted=S2(step, 'n_admitted_attack', m_, atk, p_att=pa, p_gen=pg).mean(),
                                  att_passed=S2(step, 'attack_stepup_passed', m_, atk, p_att=pa, p_gen=pg).mean(),
                                  genuine_lockout=S2(step, 'genuine_lockout_rate', m_, atk, p_att=pa, p_gen=pg).mean(),
                                  challenge=S2(step, 'online_FRR', m_, atk, p_att=pa, p_gen=pg).mean()))
st = pd.DataFrame(srows); st.to_csv(os.path.join(REV, 'stepup_realism_summary.csv'), index=False)
display(st[st.p_gen == 1.0].pivot_table(index=['attack', 'p_att'], columns='method', values='gain').round(3))
display(st[st.p_gen == 1.0].pivot_table(index=['attack', 'p_att'], columns='method', values='att_admitted').round(1))
display(st[st.p_att.isin([0.0, 0.05])].pivot_table(index=['attack', 'p_att', 'p_gen'], columns='method', values=['gain', 'genuine_lockout']).round(3))
c0g = pd.DataFrame([dict(p_gen=pg, method=m_, challenge=(S2(step, 'online_FRR', m_, 'C0_drift_only', p_gen=pg) if pg < 1 else S2(core, 'online_FRR', m_, 'C0_drift_only')).mean(),
                         lockout=(S2(step, 'genuine_lockout_rate', m_, 'C0_drift_only', p_gen=pg) if pg < 1 else S2(core, 'genuine_lockout_rate', m_, 'C0_drift_only')).mean(),
                         late_EER=(S2(step, 'ckpt_EER_late', m_, 'C0_drift_only', p_gen=pg) if pg < 1 else S2(core, 'ckpt_EER_late', m_, 'C0_drift_only')).mean())
                    for pg in P_GEN for m_ in STEP_POL]); c0g.to_csv(os.path.join(REV, 'genuine_failure_C0.csv'), index=False)
display(c0g.pivot_table(index='p_gen', columns='method', values=['challenge', 'lockout', 'late_EER']).round(3))
# where does labels-only lose its advantage? paired tests B1c - P and B1c - B1 per p_att (natural, label-seeking attacker)
step_tests = []
for atk in ['C2_zero_effort_30', 'C3_confaware_oracle_20']:
    pairs = []
    for pa in P_ATT:
        pairs += [(f'{atk} p_att={pa}: gain B1c - P', gain_step(step, 'B1c_labels_only', atk, pa, 1.0), gain_step(step, 'P_gate_DP_budget', atk, pa, 1.0)),
                  (f'{atk} p_att={pa}: gain B1c - B1', gain_step(step, 'B1c_labels_only', atk, pa, 1.0), gain_step(step, 'B1_selfupdate', atk, pa, 1.0))]
    step_tests.append(contrast_table(pairs, None, None, f'step-up realism {atk}'))
step_tests = pd.concat(step_tests, ignore_index=True); step_tests.to_csv(os.path.join(REV, 'tests_stepup.csv'), index=False)
display(step_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
# handover and selection effect
hrows = []
for m_ in HAND_POL + ['B1c_audit5']:
    for atk in ['H_handover_sessionstart', 'C3_confaware_oracle_20']:
        g_ = G2(hand, m_, atk); lo_, hi_ = boot_ci_mean(g_.values)
        hrows.append(dict(method=m_, attack=atk, gain=g_.mean(), lo=lo_, hi=hi_, att_samples=S2(hand, 'n_attack', m_, atk).mean(),
                          att_admitted=S2(hand, 'n_admitted_attack', m_, atk).mean(), challenge_C0=S2(hand, 'online_FRR', m_, 'C0_drift_only').mean(),
                          lateEER_C0=S2(hand, 'ckpt_EER_late', m_, 'C0_drift_only').mean(),
                          p_admitted_genuine=S2(hand, 'p_admitted_genuine', m_, 'C0_drift_only').mean(), p_genuine_all=S2(hand, 'p_genuine_all', m_, 'C0_drift_only').mean()))
hd = pd.DataFrame(hrows); hd.to_csv(os.path.join(REV, 'handover_selection.csv'), index=False); display(hd.round(3))
hand_tests = contrast_table([('handover gain: B1c(trust window) - B1c(per-sample)', G2(hand, 'B1c_labels_only_grace', 'H_handover_sessionstart'), G2(hand, 'B1c_labels_only', 'H_handover_sessionstart')),
                             ('handover gain: P(trust window) - P', G2(hand, 'P_gate_DP_budget_grace', 'H_handover_sessionstart'), G2(hand, 'P_gate_DP_budget', 'H_handover_sessionstart')),
                             ('selection: late EER B1c_audit5 - B1c (C0)', S2(hand, 'ckpt_EER_late', 'B1c_audit5', 'C0_drift_only'), S2(hand, 'ckpt_EER_late', 'B1c_labels_only', 'C0_drift_only')),
                             ('selection: challenge B1c_audit5 - B1c (C0)', S2(hand, 'online_FRR', 'B1c_audit5', 'C0_drift_only'), S2(hand, 'online_FRR', 'B1c_labels_only', 'C0_drift_only'))],
                            None, None, 'handover / selection')
hand_tests.to_csv(os.path.join(REV, 'tests_handover.csv'), index=False); display(hand_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
fig, ax = plt.subplots(1, 3, figsize=(7.6, 2.6), sharey=False)
for a_, atk in zip(ax[:2], ['C2_zero_effort_30', 'C3_confaware_oracle_20']):
    for m_ in STEP_POL:
        q_ = st[(st.attack == atk) & (st.p_gen == 1.0) & (st.method == m_)].sort_values('p_att')
        a_.errorbar(np.arange(len(q_)), q_.gain, yerr=[q_.gain - q_.lo, q_.hi - q_.gain], marker='o', ms=3, capsize=2, label=m_)
    a_.set_xticks(range(len(P_ATT))); a_.set_xticklabels([f'{int(p*100)}' for p in P_ATT]); a_.axhline(0, c='k', lw=.5)
    a_.set(xlabel='Attacker step-up pass rate (%)', ylabel='Attack gain', title=('(a) natural, label-seeking (C2)' if atk.startswith('C2') else '(b) crafted (C3)'))
ax[0].legend(fontsize=5)
q_ = hd[hd.attack == 'H_handover_sessionstart'].set_index('method').loc[HAND_POL]
ax[2].barh(range(len(q_)), q_.gain, xerr=[q_.gain - q_.lo, q_.hi - q_.gain], color=['C7' if 'grace' not in m else 'C3' for m in q_.index])
ax[2].set_yticks(range(len(q_))); ax[2].set_yticklabels([m.replace('_labels_only', ' lab').replace('_gate_DP_budget', '').replace('_grace', ' +window') for m in q_.index], fontsize=6)
ax[2].invert_yaxis(); ax[2].set(xlabel='Attack gain', title='(c) Handover after sign-in')
plt.tight_layout(); plt.savefig(os.path.join(FIGR, 'figR2_stepup_realism.png')); plt.show()

### 20.5 Attack suite (R1-7, R2): realisable, gate-aware, generative and stronger attackers
*C3n* reproduces each C3 vector with human motor noise (8% CV + 5 ms on hold and up-down times, down-down recomputed), i.e. what an insider can actually type on a physical keyboard. *C5* is white-box against the **whole** gate: the largest step that keeps p ≥ τ_upd **and** novelty ≤ R. *C6* samples new victim-like vectors from a diagonal Gaussian fitted to the five leaked samples (a minimal generative model) before the oracle step. *C3s* doubles both the injection rate (40%) and the burst length (20).

In [ ]:
allatk = pd.concat([core, att], ignore_index=True)
ATKS = ['C3_confaware_oracle_20', 'C3n_oracle_motornoise_20', 'C5_gateaware_20', 'C6_generative_20', 'C3s_oracle_strong_40']
arows = []
for atk in ATKS:
    for m_ in CORE:
        g_ = G2(allatk, m_, atk); lo_, hi_ = boot_ci_mean(g_.values)
        arows.append(dict(attack=atk, method=m_, gain=g_.mean(), lo=lo_, hi=hi_, admitted_frac=(S2(allatk, 'n_admitted_attack', m_, atk) / S2(allatk, 'n_attack', m_, atk)).mean()))
at = pd.DataFrame(arows); at.to_csv(os.path.join(REV, 'attack_suite_summary.csv'), index=False)
display(at.pivot_table(index='method', columns='attack', values='gain').loc[CORE, ATKS].round(3))
display(at.pivot_table(index='method', columns='attack', values='admitted_frac').loc[CORE, ATKS].round(3))
SHOW = ['B1_selfupdate', 'B1b_double_thr', 'B1c_labels_only', 'B2p_DP_selfupdate_paced', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']
lab_ = {'C3_confaware_oracle_20': 'C3 white-box', 'C3n_oracle_motornoise_20': 'C3n motor noise', 'C5_gateaware_20': 'C5 gate-aware',
        'C6_generative_20': 'C6 generative', 'C3s_oracle_strong_40': 'C3s strong (40%, x2 bursts)'}
fig, ax = plt.subplots(figsize=(7.4, 2.9)); w_ = 0.8 / len(SHOW)
for j, m_ in enumerate(SHOW):
    q_ = at[at.method == m_].set_index('attack').loc[ATKS]
    ax.bar(np.arange(len(ATKS)) + j * w_, q_.gain, w_, yerr=[q_.gain - q_.lo, q_.hi - q_.gain], capsize=1.5, label=m_, error_kw=dict(lw=.6))
ax.axhline(0, c='k', lw=.5); ax.set_xticks(np.arange(len(ATKS)) + 0.4 - w_ / 2); ax.set_xticklabels([lab_[a] for a in ATKS], fontsize=7)
ax.set(ylabel='Attack gain (PSR, 95% CI)', title='Attack gain by attacker model (CMU, 47 subjects)'); ax.legend(fontsize=5.5, ncol=4)
plt.tight_layout(); plt.savefig(os.path.join(FIGR, 'figR3_attack_suite.png')); plt.show()

### 20.6 Single-factor contrasts (R2)
Two policies were added so that each contrast changes exactly one component: **B2p** (DP self-update with the same budget *and pacing* as P) and **B3p** (the gate without noise or clipping but with P's accountant-driven update schedule). Corrected contrast map: B1c−B1 = silent admission; B3−B1 = gate (no DP); **P−B2p = gate under DP**; **P−B3p = DP noise/clipping under the gate**; B2−B1 = DP *and* budget (not DP alone, as the original text claimed); P−B2 = gate *and* pacing (the original H4b).

In [ ]:
SF = [('silent admission: B1c - B1', 'B1c_labels_only', 'B1_selfupdate'), ('gate, no DP: B3 - B1', 'B3_gate_nonDP', 'B1_selfupdate'),
      ('gate | DP, same schedule: P - B2p', 'P_gate_DP_budget', 'B2p_DP_selfupdate_paced'), ('DP | gate, same schedule: P - B3p', 'P_gate_DP_budget', 'B3p_gate_sched'),
      ('pacing | DP, no gate: B2p - B2', 'B2p_DP_selfupdate_paced', 'B2_DP_selfupdate'), ('[orig. H4b] gate+pacing: P - B2', 'P_gate_DP_budget', 'B2_DP_selfupdate'),
      ('[orig. H4c] DP+budget+pacing: P - B3', 'P_gate_DP_budget', 'B3_gate_nonDP'), ('double threshold: B1b - B1', 'B1b_double_thr', 'B1_selfupdate')]
sf_tests = []
for atk in ['C3_confaware_oracle_20', 'C3s_oracle_strong_40', 'C5_gateaware_20']:
    sf_tests.append(contrast_table([(f'{atk}: {lab}', G2(allatk, a_, atk), G2(allatk, b_, atk)) for lab, a_, b_ in SF], None, None, f'single-factor {atk}'))
sf_tests.append(contrast_table([(f'C0 challenge: {lab}', S2(core, 'online_FRR', a_, 'C0_drift_only'), S2(core, 'online_FRR', b_, 'C0_drift_only')) for lab, a_, b_ in SF], None, None, 'single-factor challenge C0'))
sf_tests = pd.concat(sf_tests, ignore_index=True); sf_tests.to_csv(os.path.join(REV, 'tests_single_factor.csv'), index=False)
display(sf_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'n_pos', 'n_neg', 'p_holm', 'interpretation']].round(4))

### 20.7 Privacy: adaptive models, non-private outputs, the user's own records, and composition (R1-4)

In [ ]:
def _mia_job(args):
    sd, v, part = args
    run = build_run(cmu, cmu_transform, part, v, part['attackers'][sid(v) % 11], DEFAULT_CFG, seed=sd)
    nonmem = [run['Z'](cmu_transform(cmu[a_]['X_raw']))[np.random.default_rng(sd).choice(400, 50, replace=False)] for a_ in part['attackers']]
    rows = []; ac_ = Accountant(DEFAULT_CFG['q'], DEFAULT_CFG['sigma'], DEFAULT_CFG['delta'])
    for mn in ['B0_static', 'B0_static_DP', 'B1_selfupdate', 'B1c_labels_only', 'B2_DP_selfupdate', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']:
        out, R_, ck_, A_ = run_stream2(run, DEFAULT_CFG, METHODS2[mn], ac_, 'C0_drift_only', seed=sd, collect_trace=True)
        mem_s = [-A_.score(U).mean() for U in run['bg_units']]; non_s = [-A_.score(U).mean() for U in nonmem]
        pg = lambda X: sigmoid(add_bias(X) @ A_.w)
        li, lo_ = -np.log(pg(run['anchor']) + 1e-12), -np.log(pg(run['cal_g']) + 1e-12)
        # template recovery: an observer of the non-private novelty statistics (mu, s) samples N(mu, s^2) and presents it
        Xsyn = np.random.default_rng(sd + 1).normal(A_.nmu, A_.nsd, size=(200, len(A_.nmu)))
        rows.append(dict(seed=sd, victim=str(v), method=mn, auc_participant=auc(np.array(mem_s), np.array(non_s)), auc_record=auc(-li, -lo_),
                         accept_template_recovery=float(np.mean(A_.score(Xsyn) >= A_.tau_acc)), accept_zero_effort=out['PSR_final'],
                         accept_genuine_late=1 - out['online_FRR_late'], n_updates=out['n_updates'], eps=out['eps_final']))
    return rows
def _mia():
    jobs = []
    for sd in SEEDS_MAIN:
        part = partition(list(cmu), 14, 6, 11, seed=sd); jobs += [(sd, v, part) for v in part['victims'][:MAX_VICTIMS]]
    with mp.get_context('fork').Pool(N_JOBS) as pool: out = pool.map(_mia_job, jobs, chunksize=1)
    return pd.DataFrame([r for rr in out for r in rr])
mia2 = cached('mia_adaptive.csv', _mia)
mrows = []
for m_ in mia2.method.unique():
    for col in ['auc_participant', 'auc_record', 'accept_template_recovery', 'accept_zero_effort']:
        s_ = mia2[mia2.method == m_].groupby('victim')[col].mean(); lo_, hi_ = boot_ci_mean(s_.values); mrows.append(dict(method=m_, metric=col, mean=s_.mean(), lo=lo_, hi=hi_))
mt = pd.DataFrame(mrows); mt.to_csv(os.path.join(REV, 'privacy_adaptive_summary.csv'), index=False)
display(mt.pivot_table(index='method', columns='metric', values='mean').round(3))
Mv = lambda m_, c_: mia2[mia2.method == m_].groupby('victim')[c_].mean()
priv_tests = contrast_table([('participant MIA: P - B3 (DP effect on adaptive)', Mv('P_gate_DP_budget', 'auc_participant'), Mv('B3_gate_nonDP', 'auc_participant')),
                             ('participant MIA: B2c - B1c', Mv('B2c_DP_labels_only', 'auc_participant'), Mv('B1c_labels_only', 'auc_participant')),
                             ('participant MIA: P - B0_static_DP (adaptation under DP)', Mv('P_gate_DP_budget', 'auc_participant'), Mv('B0_static_DP', 'auc_participant')),
                             ('record MIA (own enrolment): P - B3', Mv('P_gate_DP_budget', 'auc_record'), Mv('B3_gate_nonDP', 'auc_record')),
                             ('template recovery vs zero-effort acceptance: P', Mv('P_gate_DP_budget', 'accept_template_recovery'), Mv('P_gate_DP_budget', 'accept_zero_effort')),
                             ('template recovery vs zero-effort acceptance: B1c', Mv('B1c_labels_only', 'accept_template_recovery'), Mv('B1c_labels_only', 'accept_zero_effort'))],
                            None, None, 'privacy (adaptive)')
priv_tests.to_csv(os.path.join(REV, 'tests_privacy.csv'), index=False); display(priv_tests[['comparison', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
# update-log side channel: can an observer of update events (not of samples) tell that a victim was attacked, and when they drifted?
side = []
for m_ in ['B1_selfupdate', 'B1c_labels_only', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']:
    for col in ['n_updates', 'n_admitted']:
        a3_ = S2(core, col, m_, 'C3_confaware_oracle_20').values; a0_ = S2(core, col, m_, 'C0_drift_only').values
        side.append(dict(method=m_, observable=col, AUC_attack_vs_none=auc(a3_, a0_)))
    u_ = core[(core.method == m_) & (core.cond == 'C0_drift_only')]; b_ = ckres[(ckres.method == 'B0_static') & (ckres.cond == 'C0_drift_only')]
    rs_ = []
    for (sd, v), g_ in u_.groupby(['seed', 'victim']):
        ups = np.array([int(x) for x in str(g_.upd_per_session.iloc[0]).split(',')])
        fr_ = b_[(b_.seed == sd) & (b_.victim == v)].sort_values('sess').frr.values
        if len(ups) == len(fr_) and np.std(ups) > 0: rs_.append(scipy.stats.spearmanr(ups, fr_).statistic)
    side.append(dict(method=m_, observable='updates per session vs user drift', spearman_median=float(np.median(rs_)) if rs_ else np.nan))
side = pd.DataFrame(side); side.to_csv(os.path.join(REV, 'side_channel.csv'), index=False); display(side.round(3))
# composition across released models sharing the same background participants
comp = pd.DataFrame([dict(K=K_, eps_enrolment=ComposedAccountant(0.2, 3.0, 1e-5, K_).eps(150), eps_P_typical=ComposedAccountant(0.2, 3.0, 1e-5, K_).eps(400),
                          sigma_for_eps4_init=sigma_for_population(K_, acct.eps(150))) for K_ in [1, 5, 20, 100, 1000]])
comp.to_csv(os.path.join(REV, 'composition.csv'), index=False); display(comp.round(3))
SIG20 = sigma_for_population(20, acct.eps(150))
def _comp():
    cfg_ = dict(DEFAULT_CFG, sigma=SIG20)
    r_, _ = run_grid2(cmu, cmu_transform, SEEDS_MAIN, ['B0_static_DP', 'B2c_DP_labels_only', 'P_gate_DP_budget'], ['C0_drift_only', 'C3_confaware_oracle_20'], cfg_, (14, 6, 11), 'popDP',
                      n_jobs=N_JOBS, acct_factory=lambda c: ComposedAccountant(c['q'], c['sigma'], c['delta'], 20), max_victims=MAX_VICTIMS)
    return r_.assign(sigma=SIG20, K=20)
pop = cached('population_dp_K20.csv', _comp)
popt = pd.DataFrame([dict(method=m_, setting=lab, challenge=S2(df_, 'online_FRR', m_, 'C0_drift_only').mean(), late_EER=S2(df_, 'ckpt_EER_late', m_, 'C0_drift_only').mean(),
                          late_FAR=S2(df_, 'ckpt_FAR_late', m_, 'C0_drift_only').mean(), n_updates=S2(df_, 'n_updates', m_, 'C0_drift_only').mean(),
                          gain_C3=G2(df_, m_, 'C3_confaware_oracle_20').mean(), eps_reported=S2(df_, 'eps_final', m_, 'C0_drift_only').mean())
                     for m_ in ['B0_static_DP', 'B2c_DP_labels_only', 'P_gate_DP_budget'] for lab, df_ in [('per-model eps<=8 (sigma=3)', res), (f'population eps<=8 over K=20 (sigma={SIG20:.1f})', pop)]])
popt.to_csv(os.path.join(REV, 'population_dp_utility.csv'), index=False); display(popt.round(3))
PM = ['B0_static', 'B0_static_DP', 'B1_selfupdate', 'B1c_labels_only', 'B2_DP_selfupdate', 'B2c_DP_labels_only', 'B3_gate_nonDP', 'P_gate_DP_budget']
fig, ax = plt.subplots(1, 3, figsize=(7.6, 2.6))
for k_, (col, ttl) in enumerate([('auc_participant', '(a) Participant-level MIA'), ('auc_record', '(b) Record-level MIA (own data)')]):
    q_ = mt[mt.metric == col].set_index('method').loc[PM]
    ax[k_].errorbar(range(len(PM)), q_['mean'], yerr=[q_['mean'] - q_.lo, q_.hi - q_['mean']], fmt='o', ms=3, capsize=2)
    ax[k_].axhline(.5, c='k', ls=':', lw=.8); ax[k_].set_xticks(range(len(PM))); ax[k_].set_xticklabels(['B0', 'B0-DP', 'B1', 'B1c', 'B2', 'B2c', 'B3', 'P'], fontsize=6, rotation=45)
    ax[k_].set(title=ttl, ylabel='AUC')
ax[2].plot(comp.K, comp.eps_enrolment, 'o-', ms=3, label='enrolment (150 steps)'); ax[2].plot(comp.K, comp.eps_P_typical, 's-', ms=3, label='P typical (400 steps)')
ax[2].axhline(8, c='k', ls=':', lw=.8); ax[2].set(xscale='log', yscale='log', xlabel='Released models per participant (K)', ylabel='ε (δ=1e-5)', title='(c) Composition over K models')
ax[2].legend(fontsize=6); plt.tight_layout(); plt.savefig(os.path.join(FIGR, 'figR4_privacy.png')); plt.show()

### 20.8 External validity: drift that is not a practice effect, a non-linear verifier, touch data, on-device cost (R1-6, R2)

In [ ]:
cmu_detr, _ = drift_variant(cmu, 'detrended'); cmu_rw, rw_target = drift_variant(cmu, 'randomwalk', seed=0)
def _drift():
    parts = []
    for lab, d_ in [('detrended', cmu_detr), ('randomwalk', cmu_rw)]:
        r_, _ = run_grid2(d_, cmu_transform, SEEDS_MAIN, KEY, ['C0_drift_only', 'C3_confaware_oracle_20'], DEFAULT_CFG, (14, 6, 11), f'drift_{lab}', n_jobs=N_JOBS, max_victims=MAX_VICTIMS)
        parts.append(r_.assign(regime=lab))
    return pd.concat(parts, ignore_index=True)
drift = cached('drift_regimes.csv', _drift)
rff = cached('rff_verifier.csv', lambda: run_grid2(cmu, cmu_transform, SEEDS_MAIN, KEY, COND3, DEFAULT_CFG, (14, 6, 11), 'rff', n_jobs=N_JOBS, wrap=rff_wrap, max_victims=MAX_VICTIMS)[0])
tstrong = cached('touch_strong.csv', lambda: run_grid2(touch, touch_transform, SEEDS_MAIN, KEY, ['C0_drift_only', 'C3_confaware_oracle_20', 'C3s_oracle_strong_40'], TCFG, tsizes, 'touch_strong',
                                                         victims_fixed=touch_victims, n_jobs=N_JOBS)[0])
grow = []
for lab, df_ in [('CMU original (practice effect)', core), ('CMU drift removed', drift[drift.regime == 'detrended']), (f'CMU random-walk drift', drift[drift.regime == 'randomwalk']),
                 ('CMU, RFF verifier (257 weights)', rff), ('Touchalytics', tstrong)]:
    for m_ in KEY:
        r_ = dict(setting=lab, method=m_, challenge=S2(df_, 'online_FRR', m_, 'C0_drift_only').mean(), late_EER=S2(df_, 'ckpt_EER_late', m_, 'C0_drift_only').mean(),
                  late_FAR=S2(df_, 'ckpt_FAR_late', m_, 'C0_drift_only').mean(), gain_C3=G2(df_, m_, 'C3_confaware_oracle_20').mean())
        if 'C3s_oracle_strong_40' in set(df_.cond): r_['gain_C3s'] = G2(df_, m_, 'C3s_oracle_strong_40').mean()
        grow.append(r_)
gen = pd.DataFrame(grow); gen.to_csv(os.path.join(REV, 'generalisation_summary.csv'), index=False)
display(gen.pivot_table(index='method', columns='setting', values='challenge').loc[KEY].round(3))
display(gen.pivot_table(index='method', columns='setting', values='gain_C3').loc[KEY].round(3))
display(gen.pivot_table(index='method', columns='setting', values='gain_C3s').loc[KEY].round(3))
gen_tests = []
for lab, df_, atks in [('drift removed', drift[drift.regime == 'detrended'], ['C3_confaware_oracle_20']), ('random-walk drift', drift[drift.regime == 'randomwalk'], ['C3_confaware_oracle_20']),
                       ('RFF', rff, ['C3_confaware_oracle_20', 'C3s_oracle_strong_40']), ('Touchalytics', tstrong, ['C3_confaware_oracle_20', 'C3s_oracle_strong_40'])]:
    pairs = [(f'{lab}: challenge P - B0 (C0)', S2(df_, 'online_FRR', 'P_gate_DP_budget', 'C0_drift_only'), S2(df_, 'online_FRR', 'B0_static', 'C0_drift_only'))]
    for atk in atks:
        pairs += [(f'{lab} {atk}: gain P - B1', G2(df_, 'P_gate_DP_budget', atk), G2(df_, 'B1_selfupdate', atk)),
                  (f'{lab} {atk}: gain B1c - B1', G2(df_, 'B1c_labels_only', atk), G2(df_, 'B1_selfupdate', atk))]
    gen_tests.append(contrast_table(pairs, None, None, f'generalisation {lab}'))
gen_tests = pd.concat(gen_tests, ignore_index=True); gen_tests.to_csv(os.path.join(REV, 'tests_generalisation.csv'), index=False)
display(gen_tests[['comparison', 'n_subjects', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'interpretation']].round(4))
print(f'random-walk drift magnitude matched to original median session-8 displacement = {rw_target:.3f} session-1 SDs')
# on-device cost: corrected state size (the original 34.7 kB omitted the background negatives that on-device DP-SGD needs) and analytic FLOPs
d_ = 31; n_bg = 14 * 50; n_ref = 6 * 50
state = dict(model_and_calibration=34, novelty_stats=2 * d_, anchor=40 * d_, buffer=100 * d_, background_negatives=n_bg * d_, reference_negatives_for_calibration=n_ref * d_, verified_for_recalibration=20 * d_)
cost = pd.DataFrame([dict(component=k, floats=v, kB_float64=v * 8 / 1024, kB_float32=v * 4 / 1024) for k, v in state.items()])
cost.loc[len(cost)] = dict(component='TOTAL', floats=cost.floats.sum(), kB_float64=cost.kB_float64.sum(), kB_float32=cost.kB_float32.sum())
n_units = 14 + 140; per_step = 0.2 * (n_bg + 140) * (4 * (d_ + 1)) + n_units * 2 * (d_ + 1)   # sampled forward+backward, clip/noise
flops = dict(score=2 * (d_ + 1) + 10, dp_sgd_update_25_steps=25 * per_step, platt_recalibration=100 * (n_ref + 30) * 12)
cost.to_csv(os.path.join(REV, 'on_device_cost.csv'), index=False); display(cost.round(1)); print({k: f'{v / 1e3:.1f} kFLOP' for k, v in flops.items()})
print('A low-end Cortex-A53 core sustains O(1) GFLOP/s, so an update is O(1e-3) s of arithmetic; this is an analytic estimate, not a handset measurement.')

### 20.9 Reproducibility artefacts (R1-5, R2)

In [ ]:
parts_dump = {str(sd): {k: [str(x) for x in v] for k, v in partition(list(cmu), 14, 6, 11, seed=sd).items()} for sd in SEEDS_MAIN + SEEDS_REP + [PILOT_SEED]}
json.dump(parts_dump, open(os.path.join(REV, 'cmu_partitions_by_seed.json'), 'w'), indent=1)
json.dump(dict(default_cfg=DEFAULT_CFG, methods=METHODS2, conditions={k: list(v) for k, v in CONDITIONS2.items()}), open(os.path.join(REV, 'config_methods_conditions.json'), 'w'), indent=1, default=str)
hashes = {os.path.relpath(os.path.join(dp, f), ROOT): hashlib.sha256(open(os.path.join(dp, f), 'rb').read()).hexdigest()
          for dp, _, fs in os.walk(OUT) for f in sorted(fs) if f.endswith(('.csv', '.json'))}
hashes.update({f: hashlib.sha256(open(os.path.join(ROOT, f), 'rb').read()).hexdigest() for f in ['pacau_core.py', 'analysis_helpers.py', 'pacau_rev.py']})
json.dump(dict(generated_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), versions=versions, n_jobs=N_JOBS, sha256=hashes), open(os.path.join(REV, 'artefact_hashes.json'), 'w'), indent=1)
print(len(hashes), 'artefacts hashed ->', os.path.join(REV, 'artefact_hashes.json'))

### 20.10 Claims ledger after revision
Each claim in the revised manuscript, the evidence that now supports or limits it, and the status. Values are read from the tables above.

In [ ]:
def row(claim, evidence, status): return dict(claim=claim, evidence=evidence, status=status)
f05 = fr[np.isclose(fr.far_target, 0.05)].set_index('method'); ft05 = far_tests[far_tests.family == 'revalidated FAR 0.05'].set_index('comparison')
p0 = prim.iloc[0]; pc = prim.iloc[4]; ps = prim.iloc[5]
st1 = st[(st.p_gen == 1.0)].set_index(['attack', 'p_att', 'method']).gain
hgt = hand_tests.set_index('comparison')
sfx = sf_tests.set_index('comparison')
mtp = mt.pivot_table(index='method', columns='metric', values='mean')
ledger = pd.DataFrame([
    row('Adaptation reduces friction at a fixed probability threshold (H3)', f"challenge B0 {op.set_index('method').loc['B0_static','online_FRR']:.3f} -> P {op.set_index('method').loc['P_gate_DP_budget','online_FRR']:.3f}", 'holds, but mostly a threshold shift'),
    row('P reduces friction at matched, revalidated FAR', f"5% target: B0 {f05.loc['B0_static','challenge']:.3f}, P {f05.loc['P_gate_DP_budget','challenge']:.3f}, B1c {f05.loc['B1c_labels_only','challenge']:.3f}; P-B1c {ft05.iloc[2].mean_diff:+.3f}", 'holds (smaller); non-private adaptation reduces friction twice as much'),
    row('P matches the static verifier at matched oracle FAR', f"FRR@5%FAR P-B0 {op_tests.iloc[3].mean_diff:+.3f} [{op_tests.iloc[3].ci_lo:.3f}, {op_tests.iloc[3].ci_hi:.3f}]", 'no improvement in separability under DP'),
    row('Silent gate (with DP) reduces poisoning vs naive self-update', f"original C3 {p0.mean_diff:+.3f} (TOST p={p0.p_TOST:.3f}); pooled C3 {pc.mean_diff:+.3f} (TOST p={pc.p_TOST:.3f}); pooled C3s {ps.mean_diff:+.3f} [{ps.ci_lo:.3f}, {ps.ci_hi:.3f}]", 'small partial effect; grows with attack strength'),
    row('Labels-only updating is robust', f"C2 gain at p_att 0/5/10/20%: {st1['C2_zero_effort_30',0.0,'B1c_labels_only']:.3f}/{st1['C2_zero_effort_30',0.05,'B1c_labels_only']:.3f}/{st1['C2_zero_effort_30',0.1,'B1c_labels_only']:.3f}/{st1['C2_zero_effort_30',0.2,'B1c_labels_only']:.3f}; trust-window handover {hgt.iloc[0].mean_diff:+.3f}", 'conditional: per-sample labels and p_att <= ~5%'),
    row('Gate effect isolated under DP (P-B2p)', f"C3s {sfx.iloc[10].mean_diff:+.3f} [{sfx.iloc[10].ci_lo:.3f}, {sfx.iloc[10].ci_hi:.3f}]", 'present under strong attack; masked by pacing confound in original H4b'),
    row('DP lowers participant-level leakage of adaptive models', f"MIA AUC P {mtp.loc['P_gate_DP_budget','auc_participant']:.3f} vs B3 {mtp.loc['B3_gate_nonDP','auc_participant']:.3f}", 'holds (background cohort only)'),
    row('Non-private auxiliary state is harmless', f"samples from leaked novelty stats accepted at {mtp.loc['P_gate_DP_budget','accept_template_recovery']:.3f} (P)", 'false: novelty statistics are a usable template'),
    row('Per-model eps <= 8 is a population guarantee', f"K=20 composition: eps(enrolment) = {comp.loc[comp.K == 20, 'eps_enrolment'].iloc[0]:.1f}", 'false; per-model only'),
])
ledger.to_csv(os.path.join(REV, 'claims_ledger.csv'), index=False); display(ledger)

**References (verified; peer-reviewed unless marked preprint)**
* Killourhy, K. S. & Maxion, R. A. (2009). Comparing anomaly-detection algorithms for keystroke dynamics. *IEEE/IFIP DSN*, 125–134.
* Frank, M., Biedert, R., Ma, E., Martinovic, I. & Song, D. (2013). Touchalytics. *IEEE TIFS* 8(1), 136–148 (arXiv:1207.6231).
* Biggio, B., Didaci, L., Fumera, G. & Roli, F. (2012). Poisoning adaptive biometric systems. *SSPR&SPR*, LNCS 7626, 417–425.
* Lovisotto, G., Eberz, S. & Martinovic, I. (2020). Biometric backdoors: a poisoning attack against unsupervised template updating. *IEEE EuroS&P* (arXiv:1905.09162).
* Mhenni, A., Cherrier, E., Rosenberger, C. & Essoukri Ben Amara, N. (2019). Double serial adaptation mechanism for keystroke dynamics authentication based on a single password. *Computers & Security* 83, 151–166.
* Mhenni, A., Migdal, D., Cherrier, E., Rosenberger, C. & Essoukri Ben Amara, N. (2019). Vulnerability of adaptive strategies of keystroke dynamics based authentication against different attack types. *Cyberworlds*, 274–278.
* Pisani, P. H., Giot, R., de Carvalho, A. C. P. L. F. & Lorena, A. C. (2016). Enhanced template update: application to keystroke dynamics. *Computers & Security* 60, 134–153.
* Ma, Y., Zhu, X. & Hsu, J. (2019). Data poisoning against differentially-private learners: attacks and defenses. *IJCAI*, 4732–4738.
* Abadi, M. et al. (2016). Deep learning with differential privacy. *ACM CCS*.
* McMahan, H. B., Ramage, D., Talwar, K. & Zhang, L. (2018). Learning differentially private recurrent language models. *ICLR*.
* Baig, A. F., Eskeland, S. & Yang, B. (2023). Privacy-preserving continuous authentication using behavioral biometrics. *Int. J. Inf. Secur.* 22, 1833–1847.
* Parkinson, S., Khan, S., Liu, N. & Xu, Q. (2026). Template ageing and longitudinal verification in fixed-text keystroke dynamics. arXiv:2609.29851 (**preprint**).
* Privacy-preserving and robust mouse dynamics authentication using hybrid transformer-CNN and federated learning (2026). *Frontiers in AI*, doi:10.3389/frai.2026.1914353.
* Sitová, Z. et al. (2016). HMOG: new behavioral biometric features for continuous authentication of smartphone users. *IEEE TIFS* 11(5) (arXiv:1501.01199).
* *Added in revision:* Pisani, P. H. et al. (2019). Adaptive biometric systems: review and perspectives. *ACM Computing Surveys* 52(5), 102. doi:10.1145/3344255
* Rattani, A., Freni, B., Marcialis, G. L. & Roli, F. (2009). Template update methods in adaptive biometric systems: a critical review. *ICB 2009*, LNCS 5558, 847–856. doi:10.1007/978-3-642-01793-3_86
* Poh, N., Rattani, A. & Roli, F. (2012). Critical analysis of adaptive biometric systems. *IET Biometrics* 1(4), 179–187. doi:10.1049/iet-bmt.2012.0019
* Lovisotto, G., Eberz, S. & Martinovic, I. (2020). Biometric backdoors. *IEEE EuroS&P*, 184–197. doi:10.1109/EuroSP48549.2020.00020
* Sun, Y., Ceker, H. & Upadhyaya, S. (2016). Shared keystroke dataset for continuous authentication. *IEEE WIFS*. doi:10.1109/WIFS.2016.7823894 (request-only; not used)
* Murphy, C., Huang, J., Hou, D. & Schuckers, S. (2017). Shared dataset on natural human–computer interaction to support continuous authentication research. *IJCB*, 525–530. doi:10.1109/BTAS.2017.8272738 (request-only; not used)
* Rahimi, A. & Recht, B. (2007). Random features for large-scale kernel machines. *NeurIPS 20*.
* Lakens, D. (2017). Equivalence tests: a practical primer for t tests, correlations, and meta-analyses. *Social Psychological and Personality Science* 8(4), 355–362. doi:10.1177/1948550617697177
